In [ ]:
# =============================================================================
# CONFIDANT PRODUCTION — Layered Architecture
# =============================================================================
# Layer Stack (Bottom → Top):
#   L1: Configuration & State       L5: Profiling & Scheduling
#   L2: Compute (Model)             L6: Communication
#   L3: Dataset                     L7: Fault Tolerance
#   L4: Optimizer                   L8: Orchestration
# =============================================================================

import os
# MUST be set BEFORE import torch — CUDA reads this only once at init
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2,3,6"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import sys, time, json, math, copy, logging, threading, struct
from abc import ABC, abstractmethod
from contextlib import contextmanager
from concurrent.futures import ThreadPoolExecutor, Future
from pathlib import Path
from enum import Enum, auto
from typing import Any, Dict, List, Optional, Tuple, Callable, Set, Union
from dataclasses import dataclass, field
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader, TensorDataset
import numpy as np

# Force CUDA init NOW and validate
assert torch.cuda.is_available(), "CUDA not available!"
assert torch.cuda.device_count() == 4, f"Expected 4 GPUs, got {torch.cuda.device_count()}"
for i in range(4):
    torch.cuda.set_device(i)
    torch.cuda.reset_peak_memory_stats(i)
    print(f"  cuda:{i} ✓ {torch.cuda.get_device_name(i)}")

# Rich — terminal visualization
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.text import Text
from rich.columns import Columns
from rich.live import Live
from rich.layout import Layout
from rich import box

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)s | %(levelname)s | %(message)s',
)
logger = logging.getLogger(__name__)

# Global Rich console
CONSOLE = Console()

logger.info("Imports complete (with Rich)")


In [ ]:
# =============================================================================
# L1: CONFIGURATION & STATE
# =============================================================================
# Abstract: StateManager
# Concrete: ConfidantStateManager
# Maps to: globalStates/Common.java + Training.java + FaultTolerance.java
# =============================================================================


class StateManager(ABC):
    """Abstract state manager — shared mutable state for the training system."""

    # --- Config ---
    @abstractmethod
    def get_model_name(self) -> str: ...
    @abstractmethod
    def set_model_name(self, name: str): ...
    @abstractmethod
    def get_model_args(self) -> Dict[str, Any]: ...
    @abstractmethod
    def set_model_args(self, args: Dict[str, Any]): ...
    @abstractmethod
    def get_batch_size(self) -> int: ...
    @abstractmethod
    def set_batch_size(self, bs: int): ...

    # --- Device registry ---
    @abstractmethod
    def get_device_idx(self) -> int: ...
    @abstractmethod
    def set_device_idx(self, idx: int): ...
    @abstractmethod
    def get_worker_num(self) -> int: ...
    @abstractmethod
    def set_worker_num(self, n: int): ...
    @abstractmethod
    def get_workers(self) -> Dict[int, str]: ...
    @abstractmethod
    def set_workers(self, workers: Dict[int, str]): ...
    @abstractmethod
    def get_url_from_worker(self, idx: int) -> str: ...

    # --- Training commit (1F1B) ---
    @abstractmethod
    def init_commit(self, data_len: int, epoch: int): ...
    @abstractmethod
    def get_commit(self) -> Dict[str, Any]: ...
    @abstractmethod
    def get_commit_lock(self) -> threading.Lock: ...
    @abstractmethod
    def get_commit_condition(self) -> threading.Condition: ...

    # --- Partition ---
    @abstractmethod
    def get_partition_point(self) -> List[int]: ...
    @abstractmethod
    def set_partition_point(self, point: List[int]): ...

    # --- Semaphore ---
    @abstractmethod
    def get_semaphore(self) -> threading.Semaphore: ...

    # --- Fault tolerance state ---
    @abstractmethod
    def get_system_status(self) -> str: ...
    @abstractmethod
    def set_system_status(self, status: str): ...
    @abstractmethod
    def get_received_iter_ids(self) -> Set[int]: ...
    @abstractmethod
    def add_received_iter_id(self, iter_id: int): ...
    @abstractmethod
    def clear_received_iter_ids(self): ...


class ConfidantStateManager(StateManager):
    """Confidant state manager — maps to Common.java + Training.java + FaultTolerance.java"""

    def __init__(self):
        # Config
        self._model_name: str = ""
        self._model_args: Dict[str, Any] = {}
        self._batch_size: int = 16
        self._dataset_name: str = ""
        self._dataset_path: str = ""
        self._weights_path: str = ""

        # Device registry: idx → "host:port"
        self._device_idx: int = 0
        self._worker_num: int = 1
        self._workers: Dict[int, str] = {}

        # Training commit (1F1B tracking) — mirrors Training.java
        self._commit_lock = threading.Lock()
        self._commit_condition = threading.Condition(self._commit_lock)
        self._commit: Dict[str, Any] = {
            'forwardId': -1,
            'backwardId': -1,
            'dataLen': 0,
            'epoch': 0,
        }

        # Partition
        self._partition_point: List[int] = []

        # Semaphore (limits in-flight micro-batches)
        self._semaphore: Optional[threading.Semaphore] = None

        # Fault tolerance — mirrors FaultTolerance.java
        self._system_status: str = "NORMAL"
        self._received_iter_ids: Set[int] = set()
        self._local_replication_interval: int = 0
        self._global_replication_interval: int = 0
        self._backward_timeout: int = 30000  # ms
        self._start_iter_id: int = 0

        # Timing
        self._start_time: float = 0.0
        self._forward_times: List[float] = []
        self._backward_times: List[float] = []

    # --- Config ---
    def get_model_name(self) -> str: return self._model_name
    def set_model_name(self, name: str): self._model_name = name
    def get_model_args(self) -> Dict[str, Any]: return self._model_args
    def set_model_args(self, args: Dict[str, Any]): self._model_args = args
    def get_batch_size(self) -> int: return self._batch_size
    def set_batch_size(self, bs: int): self._batch_size = bs

    # --- Device registry ---
    def get_device_idx(self) -> int: return self._device_idx
    def set_device_idx(self, idx: int): self._device_idx = idx
    def get_worker_num(self) -> int: return self._worker_num
    def set_worker_num(self, n: int):
        self._worker_num = n
        self._semaphore = threading.Semaphore(n)
    def get_workers(self) -> Dict[int, str]: return self._workers
    def set_workers(self, workers: Dict[int, str]): self._workers = workers
    def get_url_from_worker(self, idx: int) -> str:
        return self._workers.get(idx, "")

    # --- Training commit ---
    def init_commit(self, data_len: int, epoch: int):
        with self._commit_lock:
            self._commit = {
                'forwardId': -1,
                'backwardId': -1,
                'dataLen': data_len,
                'epoch': epoch,
            }

    def get_commit(self) -> Dict[str, Any]: return self._commit
    def get_commit_lock(self) -> threading.Lock: return self._commit_lock
    def get_commit_condition(self) -> threading.Condition: return self._commit_condition

    # --- Partition ---
    def get_partition_point(self) -> List[int]: return self._partition_point
    def set_partition_point(self, point: List[int]): self._partition_point = point

    # --- Semaphore ---
    def get_semaphore(self) -> threading.Semaphore:
        if self._semaphore is None:
            self._semaphore = threading.Semaphore(self._worker_num)
        return self._semaphore

    # --- Fault tolerance ---
    def get_system_status(self) -> str: return self._system_status
    def set_system_status(self, status: str): self._system_status = status
    def get_received_iter_ids(self) -> Set[int]: return self._received_iter_ids
    def add_received_iter_id(self, iter_id: int): self._received_iter_ids.add(iter_id)
    def clear_received_iter_ids(self): self._received_iter_ids.clear()

    # --- Timing ---
    def update_forward_time(self, elapsed_ms: float): self._forward_times.append(elapsed_ms)
    def update_backward_time(self, elapsed_ms: float): self._backward_times.append(elapsed_ms)
    def set_start_time(self, t: float): self._start_time = t
    def get_start_time(self) -> float: return self._start_time

    # --- FT config ---
    def get_local_replication_interval(self) -> int: return self._local_replication_interval
    def get_global_replication_interval(self) -> int: return self._global_replication_interval
    def get_backward_timeout(self) -> int: return self._backward_timeout
    def get_start_iter_id(self) -> int: return self._start_iter_id
    def set_start_iter_id(self, v: int): self._start_iter_id = v


logger.info("L1: StateManager + ConfidantStateManager loaded") 

In [ ]:
# =============================================================================
# OBSERVABILITY: Structured Event Logger + Rich Live Dashboard
# =============================================================================
# Phase 1: Structured events (TrainingEvent, Chrome Trace, JSON export)
# Phase 2: Rich terminal live dashboard — updates in real-time during training
#
# The dashboard shows:
#   ┌─ Per-Device Status Table (current iter, phase, last latency, total fwd/bwd)
#   ├─ Live Loss Tracker (last N losses)
#   ├─ 1F1B Pipeline Progress (what each device is doing RIGHT NOW)
#   └─ Epoch Progress Bar
# =============================================================================


@dataclass
class TrainingEvent:
    """Single structured training event."""
    timestamp: float            # absolute time (time.time())
    device_id: int              # which GPU/stage
    event_type: str             # e.g. "forward", "backward", "comm_send_fwd"
    iter_id: int                # micro-batch id
    phase: str                  # "forward" or "backward" (for grouping)
    duration_ms: float          # wall-clock duration
    metadata: Dict[str, Any] = field(default_factory=dict)


# In the ConfidantLiveDashboard class, replace start(), stop(), and _refresh():

class ConfidantLiveDashboard:
    """Rich-based live terminal dashboard for pipeline-parallel training.
    
    Displays a continuously-updating panel showing per-device state,
    losses, and 1F1B pipeline progress. Works on headless servers.
    
    Thread-safety: all state updates are guarded by a lock.
    Live display refresh is rate-limited and uses try/except to avoid
    deadlocks when called from worker threads.
    """

    def __init__(self, num_devices: int = 2, num_batches: int = 10,
                 num_epochs: int = 1):
        self.num_devices = num_devices
        self.num_batches = num_batches
        self.num_epochs = num_epochs
        self.console = CONSOLE
        self._live: Optional[Live] = None
        self._lock = threading.Lock()  # Thread-safety for state updates
        self._last_refresh: float = 0.0
        self._refresh_interval: float = 0.25  # max 4 refreshes/sec
        self._use_live: bool = True  # set False for Jupyter

        # Per-device live state
        self._device_state: Dict[int, Dict[str, Any]] = {
            d: {
                "current_phase": "idle",
                "current_iter": -1,
                "last_fwd_ms": 0.0,
                "last_bwd_ms": 0.0,
                "total_fwd": 0,
                "total_bwd": 0,
                "avg_fwd_ms": 0.0,
                "avg_bwd_ms": 0.0,
                "fwd_times": [],
                "bwd_times": [],
            }
            for d in range(num_devices)
        }

        # Loss tracking
        self._losses: List[Tuple[int, float]] = []
        self._max_losses = 20

        # Epoch tracking
        self._current_epoch: int = 0
        self._completed_iters: int = 0

        # Detect Jupyter — Live doesn't work well there
        self._in_jupyter = self._detect_jupyter()
        if self._in_jupyter:
            self._use_live = False

    @staticmethod
    def _detect_jupyter() -> bool:
        try:
            from IPython import get_ipython
            shell = get_ipython()
            if shell is None:
                return False
            return shell.__class__.__name__ in ('ZMQInteractiveShell', 'TerminalInteractiveShell')
        except (ImportError, NameError):
            return False

    def start(self):
        """Start the live display (skipped in Jupyter)."""
        if not self._use_live:
            self.console.print(Panel(
                "[bold cyan]Dashboard: Jupyter mode — summary printed after training[/]",
                border_style="yellow",
            ))
            return
        try:
            self._live = Live(
                self._build_display(),
                console=self.console,
                refresh_per_second=4,
                transient=False,
            )
            self._live.start()
        except Exception as e:
            logger.warning(f"Dashboard Live start failed: {e}, falling back to static mode")
            self._live = None
            self._use_live = False

    def stop(self):
        """Stop the live display and print final state."""
        if self._live:
            try:
                self._live.stop()
            except Exception:
                pass
            self._live = None

        # Always print final snapshot
        try:
            self.console.print(self._build_display())
        except Exception:
            pass

    def update_forward(self, device_id: int, iter_id: int, duration_ms: float):
        with self._lock:
            state = self._device_state[device_id]
            state["current_phase"] = "forward"
            state["current_iter"] = iter_id
            state["last_fwd_ms"] = duration_ms
            state["total_fwd"] += 1
            state["fwd_times"].append(duration_ms)
            state["avg_fwd_ms"] = sum(state["fwd_times"][-20:]) / min(len(state["fwd_times"]), 20)
        self._refresh()

    def update_backward(self, device_id: int, iter_id: int, duration_ms: float):
        with self._lock:
            state = self._device_state[device_id]
            state["current_phase"] = "backward"
            state["current_iter"] = iter_id
            state["last_bwd_ms"] = duration_ms
            state["total_bwd"] += 1
            state["bwd_times"].append(duration_ms)
            state["avg_bwd_ms"] = sum(state["bwd_times"][-20:]) / min(len(state["bwd_times"]), 20)
            self._completed_iters = max(self._completed_iters, iter_id + 1)
        self._refresh()

    def update_loss(self, iter_id: int, loss: float):
        with self._lock:
            self._losses.append((iter_id, loss))
            if len(self._losses) > self._max_losses:
                self._losses = self._losses[-self._max_losses:]
        self._refresh()

    def update_epoch(self, epoch: int):
        with self._lock:
            self._current_epoch = epoch
            self._completed_iters = 0
            for d in range(self.num_devices):
                self._device_state[d]["total_fwd"] = 0
                self._device_state[d]["total_bwd"] = 0
                self._device_state[d]["fwd_times"].clear()
                self._device_state[d]["bwd_times"].clear()
        self._refresh()

    def update_comm(self, device_id: int, iter_id: int):
        with self._lock:
            self._device_state[device_id]["current_phase"] = "comm"
            self._device_state[device_id]["current_iter"] = iter_id
        self._refresh()

    def _refresh(self):
        """Rate-limited, thread-safe refresh."""
        now = time.time()
        if now - self._last_refresh < self._refresh_interval:
            return
        self._last_refresh = now

        if self._live:
            try:
                with self._lock:
                    display = self._build_display()
                self._live.update(display)
            except Exception:
                pass  # Never let dashboard crash training

    def _build_display(self) -> Panel:
        """Build the full dashboard layout. Must be called with self._lock held."""
        # ── 1. Device Status Table ──
        device_table = Table(
            title=f"[bold cyan]Epoch {self._current_epoch + 1}/{self.num_epochs}  "
                  f"│  Batch {self._completed_iters}/{self.num_batches}[/]",
            box=box.ROUNDED,
            show_lines=True,
            expand=True,
        )
        device_table.add_column("GPU", style="bold white", width=5, justify="center")
        device_table.add_column("Phase", width=10, justify="center")
        device_table.add_column("Iter", width=5, justify="center")
        device_table.add_column("Fwd (ms)", width=10, justify="right")
        device_table.add_column("Bwd (ms)", width=10, justify="right")
        device_table.add_column("Avg Fwd", width=10, justify="right")
        device_table.add_column("Avg Bwd", width=10, justify="right")
        device_table.add_column("# Fwd", width=6, justify="right")
        device_table.add_column("# Bwd", width=6, justify="right")

        phase_colors = {
            "forward": "green", "backward": "red",
            "comm": "yellow", "idle": "dim",
        }

        for d in range(self.num_devices):
            s = self._device_state[d]
            phase = s["current_phase"]
            color = phase_colors.get(phase, "white")
            device_table.add_row(
                f"{d}",
                f"[{color}]{phase}[/]",
                f"{s['current_iter']}" if s['current_iter'] >= 0 else "—",
                f"{s['last_fwd_ms']:.2f}" if s['last_fwd_ms'] > 0 else "—",
                f"{s['last_bwd_ms']:.2f}" if s['last_bwd_ms'] > 0 else "—",
                f"{s['avg_fwd_ms']:.2f}" if s['avg_fwd_ms'] > 0 else "—",
                f"{s['avg_bwd_ms']:.2f}" if s['avg_bwd_ms'] > 0 else "—",
                f"{s['total_fwd']}", f"{s['total_bwd']}",
            )

        # ── 2. Loss Sparkline ──
        loss_text = Text()
        if self._losses:
            loss_text.append("Loss: ", style="bold white")
            recent = self._losses[-10:]
            for i, (it, loss) in enumerate(recent):
                if i > 0 and loss < recent[i-1][1]:
                    loss_text.append(f"{loss:.4f} ", style="green")
                elif i > 0 and loss > recent[i-1][1]:
                    loss_text.append(f"{loss:.4f} ", style="red")
                else:
                    loss_text.append(f"{loss:.4f} ", style="yellow")
        else:
            loss_text.append("Loss: waiting...", style="dim")

        # ── 3. Progress Bar ──
        total = self.num_batches * self.num_epochs
        done = self._current_epoch * self.num_batches + self._completed_iters
        pct = done / total if total > 0 else 0
        bar_w = 40
        filled = int(bar_w * pct)
        bar = f"[green]{'█' * filled}[/][dim]{'░' * (bar_w - filled)}[/] {pct*100:.0f}%"
        progress_text = Text.from_markup(f"Progress: {bar}  ({done}/{total} batches)")

        # ── 4. Pipeline Viz ──
        pipeline_text = self._build_pipeline_viz()

        from rich.console import Group
        content = Group(device_table, Text(""), loss_text, progress_text,
                        Text(""), pipeline_text)

        return Panel(
            content,
            title="[bold bright_white]🔧 CONFIDANT Pipeline Dashboard[/]",
            border_style="bright_blue",
            expand=True,
        )

    def _build_pipeline_viz(self) -> Text:
        text = Text()
        text.append("1F1B Pipeline:\n", style="bold underline")
        for d in range(self.num_devices):
            s = self._device_state[d]
            text.append(f"  GPU {d}: ", style="bold")
            total_fwd = s["total_fwd"]
            total_bwd = s["total_bwd"]
            max_show = min(15, self.num_batches)
            for i in range(max_show):
                if i < total_fwd and i < total_bwd:
                    text.append("FB ", style="bright_green")
                elif i < total_fwd:
                    text.append("F· ", style="green")
                elif i < total_bwd:
                    text.append("·B ", style="red")
                else:
                    text.append("·· ", style="dim")
            text.append("\n")
        return text


class ConfidantEventLogger:
    """Thread-safe structured event logger with optional Rich live dashboard.
    
    Usage:
        el = ConfidantEventLogger()
        el.enable_dashboard(num_devices=2, num_batches=10)
        with el.log_event(device_id=0, event_type="forward", iter_id=5, phase="forward"):
            output = model(input)
        
        # After training:
        el.to_chrome_trace("trace.json")
        el.print_rich_summary()
    """

    def __init__(self):
        self._events: List[TrainingEvent] = []
        self._lock = threading.Lock()
        self._epoch_start_time: float = 0.0
        self._enabled: bool = True
        self._dashboard: Optional[ConfidantLiveDashboard] = None

    def enable_dashboard(self, num_devices: int = 2, num_batches: int = 10,
                         num_epochs: int = 1):
        """Create and attach a live dashboard."""
        self._dashboard = ConfidantLiveDashboard(
            num_devices=num_devices,
            num_batches=num_batches,
            num_epochs=num_epochs,
        )

    def start_dashboard(self):
        if self._dashboard:
            self._dashboard.start()

    def stop_dashboard(self):
        if self._dashboard:
            self._dashboard.stop()

    def set_epoch_start(self, t: float):
        self._epoch_start_time = t

    def enable(self):
        self._enabled = True

    def disable(self):
        self._enabled = False

    @contextmanager
    def log_event(self, device_id: int, event_type: str, iter_id: int,
                  phase: str = "", **metadata):
        """Context manager that times a block and records a TrainingEvent."""
        if not self._enabled:
            yield
            return

        start = time.time()
        yield
        duration_ms = (time.time() - start) * 1000

        event = TrainingEvent(
            timestamp=start,
            device_id=device_id,
            event_type=event_type,
            iter_id=iter_id,
            phase=phase,
            duration_ms=duration_ms,
            metadata=metadata,
        )
        with self._lock:
            self._events.append(event)

        # Update live dashboard
        self._push_to_dashboard(event)

    def record_event(self, device_id: int, event_type: str, iter_id: int,
                     phase: str, duration_ms: float, timestamp: Optional[float] = None,
                     **metadata):
        """Record a pre-measured event."""
        if not self._enabled:
            return
        event = TrainingEvent(
            timestamp=timestamp or time.time(),
            device_id=device_id,
            event_type=event_type,
            iter_id=iter_id,
            phase=phase,
            duration_ms=duration_ms,
            metadata=metadata,
        )
        with self._lock:
            self._events.append(event)

        self._push_to_dashboard(event)

    def _push_to_dashboard(self, event: TrainingEvent):
        """Route events to the live dashboard."""
        if not self._dashboard:
            return
        d = self._dashboard
        t = event.event_type

        if t in ("forward",):
            d.update_forward(event.device_id, event.iter_id, event.duration_ms)
        elif t == "forward_last":
            d.update_forward(event.device_id, event.iter_id, event.duration_ms)
            loss = event.metadata.get("loss")
            if loss is not None:
                d.update_loss(event.iter_id, loss)
        elif t == "backward":
            d.update_backward(event.device_id, event.iter_id, event.duration_ms)
        elif t.startswith("comm_"):
            d.update_comm(event.device_id, event.iter_id)
        elif t == "epoch_end":
            pass  # handled by orchestrator

    def get_events(self) -> List[TrainingEvent]:
        with self._lock:
            return list(self._events)

    def clear(self):
        with self._lock:
            self._events.clear()

    # ── Export: Chrome Trace ──────────────────────────────────────────────

    def to_chrome_trace(self, filepath: str = "1f1b_trace.json"):
        """Export as Chrome Trace Event format for chrome://tracing or perfetto."""
        trace_events = []
        t0 = self._epoch_start_time or (
            min(e.timestamp for e in self._events) if self._events else 0
        )

        for ev in self._events:
            trace_events.append({
                "name": f"{ev.event_type} i{ev.iter_id}",
                "cat": ev.phase or ev.event_type,
                "ph": "X",
                "ts": (ev.timestamp - t0) * 1e6,
                "dur": ev.duration_ms * 1000,
                "pid": 0,
                "tid": ev.device_id,
                "args": {
                    "iter_id": ev.iter_id,
                    "device_id": ev.device_id,
                    "duration_ms": round(ev.duration_ms, 3),
                    **ev.metadata,
                },
            })

        with open(filepath, 'w') as f:
            json.dump({"traceEvents": trace_events}, f, indent=2)
        logger.info(f"Chrome trace exported → {filepath} ({len(trace_events)} events)")

    # ── Export: Raw JSON ──────────────────────────────────────────────────

    def to_json(self, filepath: str = "1f1b_events.json"):
        events = []
        for ev in self._events:
            events.append({
                "timestamp": ev.timestamp,
                "relative_ms": (ev.timestamp - self._epoch_start_time) * 1000
                    if self._epoch_start_time else 0,
                "device_id": ev.device_id,
                "event_type": ev.event_type,
                "iter_id": ev.iter_id,
                "phase": ev.phase,
                "duration_ms": round(ev.duration_ms, 4),
                **ev.metadata,
            })
        with open(filepath, 'w') as f:
            json.dump(events, f, indent=2)
        logger.info(f"Events exported → {filepath} ({len(events)} events)")

    # ── Rich: Summary Table ───────────────────────────────────────────────

    def print_rich_summary(self):
        """Print a Rich-formatted summary table to the console."""
        if not self._events:
            CONSOLE.print("[dim]No events recorded.[/]")
            return

        # Stats by (device, event_type)
        stats: Dict[Tuple[int, str], Dict[str, float]] = defaultdict(
            lambda: {"count": 0, "total_ms": 0.0, "min_ms": float('inf'), "max_ms": 0.0}
        )
        for ev in self._events:
            key = (ev.device_id, ev.event_type)
            s = stats[key]
            s["count"] += 1
            s["total_ms"] += ev.duration_ms
            s["min_ms"] = min(s["min_ms"], ev.duration_ms)
            s["max_ms"] = max(s["max_ms"], ev.duration_ms)

        table = Table(
            title="[bold]Observability Summary[/]",
            box=box.DOUBLE_EDGE,
            show_lines=True,
        )
        table.add_column("GPU", style="cyan bold", justify="center", width=5)
        table.add_column("Event Type", style="white", width=22)
        table.add_column("Count", style="yellow", justify="right", width=6)
        table.add_column("Total (ms)", justify="right", width=11)
        table.add_column("Avg (ms)", justify="right", width=10)
        table.add_column("Min (ms)", style="green", justify="right", width=10)
        table.add_column("Max (ms)", style="red", justify="right", width=10)

        for (dev, etype), s in sorted(stats.items()):
            avg = s["total_ms"] / s["count"] if s["count"] else 0
            table.add_row(
                str(dev), etype, str(s["count"]),
                f"{s['total_ms']:.1f}", f"{avg:.2f}",
                f"{s['min_ms']:.2f}", f"{s['max_ms']:.2f}",
            )

        CONSOLE.print(table)

    # ── Rich: 1F1B Timeline ───────────────────────────────────────────────

    def print_rich_1f1b_timeline(self, max_iters: int = 20):
        """Print a Rich-formatted 1F1B pipeline timeline."""
        if not self._events:
            CONSOLE.print("[dim]No events recorded.[/]")
            return

        compute_events = [e for e in self._events
                          if e.event_type in ("forward", "backward", "forward_last")]
        if not compute_events:
            CONSOLE.print("[dim]No compute events recorded.[/]")
            return

        devices = sorted(set(e.device_id for e in compute_events))
        all_iters = sorted(set(e.iter_id for e in compute_events))
        cap = min(max_iters, len(all_iters))
        show_iters = all_iters[:cap]

        fwd_lookup = {}
        bwd_lookup = {}
        for e in compute_events:
            if e.event_type in ("forward", "forward_last"):
                fwd_lookup[(e.device_id, e.iter_id)] = e
            elif e.event_type == "backward":
                bwd_lookup[(e.device_id, e.iter_id)] = e

        table = Table(
            title=f"[bold]1F1B Pipeline Timeline (first {cap} iters)[/]",
            box=box.SIMPLE_HEAVY,
            show_lines=False,
            padding=(0, 1),
        )
        table.add_column("Device", style="bold cyan", width=10)
        for it in show_iters:
            table.add_column(f"i{it}", width=4, justify="center")

        for dev in devices:
            # Forward row
            fwd_cells = []
            for it in show_iters:
                if (dev, it) in fwd_lookup:
                    fwd_cells.append("[green bold]F[/]")
                else:
                    fwd_cells.append("[dim]·[/]")
            table.add_row(f"GPU {dev} Fwd", *fwd_cells)

            # Backward row
            bwd_cells = []
            for it in show_iters:
                if (dev, it) in bwd_lookup:
                    bwd_cells.append("[red bold]B[/]")
                else:
                    bwd_cells.append("[dim]·[/]")
            table.add_row(f"GPU {dev} Bwd", *bwd_cells)

        CONSOLE.print(table)

    # ── Rich: Per-Iteration Breakdown ─────────────────────────────────────

    def print_rich_iteration_breakdown(self, iter_ids: Optional[List[int]] = None,
                                       max_iters: int = 5):
        """Print per-iteration latency breakdown with Rich formatting."""
        if not self._events:
            CONSOLE.print("[dim]No events recorded.[/]")
            return

        all_iters = sorted(set(e.iter_id for e in self._events if e.iter_id >= 0))
        show = iter_ids if iter_ids else all_iters[:max_iters]

        table = Table(
            title="[bold]Per-Iteration Breakdown[/]",
            box=box.ROUNDED,
            show_lines=True,
        )
        table.add_column("Iter", style="bold yellow", justify="center", width=5)
        table.add_column("GPU", style="cyan", justify="center", width=5)
        table.add_column("Event", style="white", width=22)
        table.add_column("Duration (ms)", justify="right", width=13)

        for it in show:
            it_events = sorted(
                [e for e in self._events if e.iter_id == it],
                key=lambda x: x.timestamp
            )
            for e in it_events:
                color = "green" if e.phase == "forward" else "red" if e.phase == "backward" else "yellow"
                table.add_row(
                    str(it), str(e.device_id), e.event_type,
                    f"[{color}]{e.duration_ms:.2f}[/]",
                )

        CONSOLE.print(table)

    # ── Legacy plain-text methods (kept for compatibility) ────────────────

    def summary(self) -> str:
        if not self._events:
            return "No events recorded."
        stats: Dict[Tuple[int, str], Dict[str, float]] = defaultdict(
            lambda: {"count": 0, "total_ms": 0.0, "min_ms": float('inf'), "max_ms": 0.0}
        )
        for ev in self._events:
            key = (ev.device_id, ev.event_type)
            s = stats[key]
            s["count"] += 1
            s["total_ms"] += ev.duration_ms
            s["min_ms"] = min(s["min_ms"], ev.duration_ms)
            s["max_ms"] = max(s["max_ms"], ev.duration_ms)
        lines = []
        header = (f"{'Device':>6} | {'Event Type':<25} | {'Count':>5} | "
                  f"{'Total(ms)':>10} | {'Avg(ms)':>8} | {'Min(ms)':>8} | {'Max(ms)':>8}")
        lines.append(header)
        lines.append("─" * len(header))
        for (dev, etype), s in sorted(stats.items()):
            avg = s["total_ms"] / s["count"] if s["count"] else 0
            lines.append(
                f"{dev:>6} | {etype:<25} | {s['count']:>5} | "
                f"{s['total_ms']:>10.1f} | {avg:>8.2f} | {s['min_ms']:>8.2f} | {s['max_ms']:>8.2f}"
            )
        return "\n".join(lines)

    def print_1f1b_timeline(self, max_iters: int = 20):
        """ASCII fallback timeline."""
        if not self._events:
            print("No events recorded.")
            return
        compute_events = [e for e in self._events
                          if e.event_type in ("forward", "backward", "forward_last")]
        if not compute_events:
            print("No compute events recorded.")
            return
        devices = sorted(set(e.device_id for e in compute_events))
        all_iters = sorted(set(e.iter_id for e in compute_events))
        cap = min(max_iters, len(all_iters))
        show_iters = all_iters[:cap]
        fwd_lookup = {}
        bwd_lookup = {}
        for e in compute_events:
            if e.event_type in ("forward", "forward_last"):
                fwd_lookup[(e.device_id, e.iter_id)] = e
            elif e.event_type == "backward":
                bwd_lookup[(e.device_id, e.iter_id)] = e
        col_w = 5
        print(f"\n┌─ 1F1B Pipeline Timeline (first {cap} iterations) ─{'─' * (cap * col_w)}")
        for dev in devices:
            fwd_parts = []
            for it in show_iters:
                if (dev, it) in fwd_lookup:
                    tag = f"F{it:02d}" if it < 100 else f"F{it}"
                    fwd_parts.append(f"{tag:>{col_w - 1}} ")
                else:
                    fwd_parts.append(f"{'·':>{col_w - 1}} ")
            print(f"│ Dev {dev} Fwd: {''.join(fwd_parts)}")
            bwd_parts = []
            for it in show_iters:
                if (dev, it) in bwd_lookup:
                    tag = f"B{it:02d}" if it < 100 else f"B{it}"
                    bwd_parts.append(f"{tag:>{col_w - 1}} ")
                else:
                    bwd_parts.append(f"{'·':>{col_w - 1}} ")
            print(f"│ Dev {dev} Bwd: {''.join(bwd_parts)}")
            if dev < devices[-1]:
                print(f"│{'':>{12}}{'─' * (cap * col_w)}")
        print(f"└─{'─' * (12 + cap * col_w)}")

    def print_iteration_breakdown(self, iter_ids: Optional[List[int]] = None,
                                  max_iters: int = 5):
        if not self._events:
            print("No events recorded.")
            return
        all_iters = sorted(set(e.iter_id for e in self._events if e.iter_id >= 0))
        show = iter_ids if iter_ids else all_iters[:max_iters]
        print(f"\n┌─ Per-Iteration Breakdown ─────────────────────────────────────")
        for it in show:
            it_events = [e for e in self._events if e.iter_id == it]
            if not it_events:
                continue
            print(f"│ Iter {it}:")
            for e in sorted(it_events, key=lambda x: x.timestamp):
                print(f"│   Dev {e.device_id:>2} │ {e.event_type:<22} │ {e.duration_ms:>8.2f} ms")
            print(f"│{'─' * 60}")
        print(f"└──────────────────────────────────────────────────────────────")


# ── Global singleton ──────────────────────────────────────────────────────
EVENT_LOGGER = ConfidantEventLogger()

logger.info("EventLogger + ConfidantLiveDashboard loaded (Phase 1+2 Observability)")

In [ ]:
# =============================================================================
# L2: COMPUTE (MODEL)
# =============================================================================
# Abstract: ComputeBackend
# Concrete: ConfidantComputeBackend
# Maps to: Model.java + train.cpp (trainForward, trainIntermediate,
#          trainIntermediateLast, trainBackwardCentral, trainBackwardWorker)
# =============================================================================


class ComputeBackend(ABC):
    """Abstract compute backend — model creation, forward, backward with caching."""

    @abstractmethod
    def create_sub_model(self, full_model: nn.Module, partition_points: List[int],
                         device_id: int, num_devices: int, device: torch.device) -> nn.Module: ...

    @abstractmethod
    def init_train(self, batch_size: int, device_num: int, device_idx: int): ...

    @abstractmethod
    def init_train_epoch(self): ...

    @abstractmethod
    def train_forward(self, iter_id: int, input_data: torch.Tensor
                      ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]: ...

    @abstractmethod
    def train_intermediate(self, iter_id: int, input_data: torch.Tensor
                           ) -> torch.Tensor: ...

    @abstractmethod
    def train_intermediate_last(self, iter_id: int, input_data: torch.Tensor,
                                labels: torch.Tensor) -> Tuple[float, torch.Tensor]: ...

    @abstractmethod
    def train_backward_central(self, iter_id: int, grad: torch.Tensor): ...

    @abstractmethod
    def train_backward_worker(self, iter_id: int, grad: torch.Tensor
                              ) -> torch.Tensor: ...


class ConfidantComputeBackend(ComputeBackend):
    """PyTorch compute backend — matches train.cpp logic exactly.
    
    Activation caching per iter_id mirrors CommonStates::storeIntermediate / getIntermediate.
    Weight versioning mirrors OptimizerWeightVersion.
    
    defer_step modes:
      True:  Gradients accumulate across micro-batches. The orchestrator
             calls zero_all_grads() at epoch start and step_optimizer()
             at epoch end.
      False: Per-micro-batch zero_grad / step cycle (mirrors Java Confidant).
             Backward methods recompute the forward pass from stored inputs
             to obtain a fresh autograd graph, since earlier micro-batches'
             optimizer.step() invalidates stored activation graphs.
    """

    def __init__(self, device: torch.device, device_id: int = 0,
                 event_logger: Optional[ConfidantEventLogger] = None,
                 defer_step: bool = True):
        self.device = device
        self.device_id = device_id
        self.event_logger = event_logger
        self.defer_step = defer_step
        self.sub_model: Optional[nn.Module] = None
        self.optimizer_ref = None  # Set by L4 after creation

        # Activation cache: iter_id → {0: input, 1: output}
        # Matches CommonStates::storeIntermediate(iterId, tensor, slot)
        self.intermediates: Dict[int, Dict[int, torch.Tensor]] = {}

        # Weight versioning: iter_id → version
        self.weight_versions: Dict[int, int] = {}
        self.current_version: int = 0
        self.accumulation_steps: int = 1
    # ── Gradient accumulation helpers ──────────────────────────────────────

    def zero_all_grads(self):
        """Zero gradients on the optimizer. Called once at epoch start.
        set_to_none=True deallocates grad tensors instead of zeroing in-place.
        """
        if self.optimizer_ref is not None:
            self.optimizer_ref.zero_grad(set_to_none=True)

    def step_optimizer(self):
        """Step the optimizer (applies accumulated gradients). Called at epoch end."""
        if self.optimizer_ref is not None:
            self.optimizer_ref.step()
            self.current_version += 1

    # ── Model creation ────────────────────────────────────────────────────

    def create_sub_model(self, full_model: nn.Module, partition_points: List[int],
                         device_id: int, num_devices: int, device: torch.device) -> nn.Module:
        """Extract sub-model from full model based on partition points."""
        self.device = device

        if not hasattr(full_model, 'layers'):
            raise ValueError("Model must have a 'layers' attribute (nn.ModuleList)")

        num_layers = len(full_model.layers)

        # 1. Calculate layer range
        if device_id == 0:
            start, end = 0, partition_points[0] if partition_points else num_layers - 1
        elif device_id < len(partition_points):
            start, end = partition_points[device_id - 1] + 1, partition_points[device_id]
        else:
            start, end = partition_points[-1] + 1 if partition_points else 0, num_layers - 1

        sub_layers = list(full_model.layers[start:end + 1])
        
        is_first = (device_id == 0)
        is_last = (device_id == num_devices - 1)

        # 2. Define SubModel with correct attribute names
        class SubModel(nn.Module):
            def __init__(self, layers, pos_emb, ln_f, classifier, is_first_stage, is_last_stage):
                super().__init__()
                self.layers = nn.Sequential(*layers)
                self.pos_emb = pos_emb 
                self.ln_f = ln_f       
                self.classifier = classifier 
                self.is_first_stage = is_first_stage
                self.is_last_stage = is_last_stage # <--- This fixes the AttributeError

            def forward(self, x):
                # First stage positional refinement
                if self.is_first_stage and self.pos_emb is not None:
                    B, T, C = x.shape
                    pos_ids = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
                    x = x + self.pos_emb(pos_ids)
                
                # Transformer blocks
                for layer in self.layers:
                    x = layer(x)
                
                # Last stage final normalization and classification
                if self.is_last_stage:
                    if self.ln_f is not None:
                        x = self.ln_f(x)
                    if self.classifier is not None:
                        x = self.classifier(x.mean(dim=1))
                return x

        # 3. Extract components
        pos_emb = full_model.pos_emb if is_first else None
        ln_f = full_model.ln_f if is_last else None
        classifier = full_model.classifier if (is_last and hasattr(full_model, 'classifier')) else None

        self.sub_model = SubModel(
            sub_layers, pos_emb, ln_f, classifier, is_first, is_last
        ).to(device)

        logger.info(f"SubModel Stage {device_id}: layers {start}-{end}, "
                    f"is_last_stage={is_last}, device={device}")
        
        return self.sub_model

    def init_train(self, batch_size: int, device_num: int, device_idx: int):
        """Maps to initTrain() in train.cpp"""
        pass  # Config already managed by StateManager

    def init_train_epoch(self):
        """Maps to initTrainEpoch() in train.cpp — clear caches, reset model."""
        self.intermediates.clear()
        self.weight_versions.clear()
        if self.sub_model is not None:
            self.sub_model.train()
        torch.cuda.empty_cache()

    # ── Forward ───────────────────────────────────────────────────────────

    def train_forward(self, iter_id: int, input_data: torch.Tensor
                      ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """Central node forward — maps to trainForward() in train.cpp.
        
        MEMORY FIX: The output returned to the comm layer is detached
        (breaks autograd chain for the downstream stage). The live
        output (with grad_fn) is kept ONLY in self.intermediates for
        this stage's own backward pass, and deleted immediately after.
        """
        assert self.sub_model is not None
        el = self.event_logger

        # Detach input to break autograd chain from previous batches
        input_data = input_data.detach().requires_grad_(True)

        if el:
            with el.log_event(self.device_id, "forward", iter_id, phase="forward",
                              version=self.current_version):
                self.weight_versions[iter_id] = self.current_version
                output = self.sub_model(input_data)
                if self.defer_step:
                    self.intermediates[iter_id] = {0: input_data, 1: output}
                else:
                    # defer_step=False: don't store live output — its autograd
                    # graph goes stale after optimizer.step(). Store only the
                    # detached input; backward will recompute forward.
                    self.intermediates[iter_id] = {0: input_data.detach()}
        else:
            self.weight_versions[iter_id] = self.current_version
            output = self.sub_model(input_data)
            if self.defer_step:
                self.intermediates[iter_id] = {0: input_data, 1: output}
            else:
                self.intermediates[iter_id] = {0: input_data.detach()}

        # Return DETACHED output — downstream stage doesn't need this stage's graph
        return output.detach(), None

    def train_intermediate(self, iter_id: int, input_data: torch.Tensor) -> torch.Tensor:
        """Non-last worker forward — maps to trainIntermediate() in train.cpp
        
        MEMORY FIX: Returns detached output. Live tensor kept only in
        intermediates for this stage's backward.
        """
        assert self.sub_model is not None
        el = self.event_logger

        # Detach input to break autograd chain from previous batches
        input_data = input_data.detach().requires_grad_(True)

        if el:
            with el.log_event(self.device_id, "forward", iter_id, phase="forward",
                              version=self.current_version):
                self.weight_versions[iter_id] = self.current_version
                output = self.sub_model(input_data)
                if self.defer_step:
                    self.intermediates[iter_id] = {0: input_data, 1: output}
                else:
                    self.intermediates[iter_id] = {0: input_data.detach()}
        else:
            self.weight_versions[iter_id] = self.current_version
            output = self.sub_model(input_data)
            if self.defer_step:
                self.intermediates[iter_id] = {0: input_data, 1: output}
            else:
                self.intermediates[iter_id] = {0: input_data.detach()}

        # Return DETACHED output — downstream stage creates its own graph
        return output.detach()

    def train_intermediate_last(self, iter_id: int, input_data: torch.Tensor,
                                labels: torch.Tensor) -> Tuple[float, torch.Tensor]:
        """Last worker: forward + loss + backward — maps to trainIntermediateLast() in train.cpp.
        Returns (loss_value, input_gradient).
        
        MEMORY FIX: When defer_step=True, we do NOT step or zero_grad here.
        Gradients accumulate. The orchestrator steps once at epoch end.
        This is REQUIRED for pipeline parallelism: multiple micro-batches
        are in-flight, and stepping would modify weights that later
        micro-batches' stored activations depend on.
        """
        assert self.sub_model is not None
        el = self.event_logger

        def _do_last(input_data, labels):
            input_data = input_data.detach().requires_grad_(True)
            if iter_id < 5 or iter_id % 50 == 0:
                param = next(self.sub_model.parameters())
                p_sum = param.sum().item()
                # Use print/logger as available in your context
                print(f"DEBUG: Batch {iter_id} - Weight Sum: {p_sum:.6f}")
            # Time forward separately
            fwd_start = time.time()
            output = self.sub_model(input_data)
            labels_dev = labels.to(self.device)
            loss = F.cross_entropy(output, labels_dev) / self.accumulation_steps
            loss_val = loss.item() * self.accumulation_steps # Scale back up for logging
            fwd_ms = (time.time() - fwd_start) * 1000

            if not self.defer_step and self.optimizer_ref is not None:
                self.optimizer_ref.zero_grad(set_to_none=True)
            
            # Time backward separately
            bwd_start = time.time()
            loss.backward()
            bwd_ms = (time.time() - bwd_start) * 1000
            
            if not self.defer_step and self.optimizer_ref is not None:
                self.optimizer_ref.step()
                self.current_version += 1

            # MEMORY FIX: detach the gradient — we only need the data, not the graph
            input_grad = input_data.grad.detach().clone() if input_data.grad is not None else torch.zeros_like(input_data)
            
            # MEMORY FIX: explicitly free forward graph and labels
            del output, loss, labels_dev
            
            return loss_val, input_grad, fwd_ms, bwd_ms

        if el:
            loss_val, input_grad, fwd_ms, bwd_ms = _do_last(input_data, labels)
            el.record_event(self.device_id, "forward_last", iter_id, phase="forward",
                           duration_ms=fwd_ms, version=self.current_version, loss=loss_val)
            el.record_event(self.device_id, "backward", iter_id, phase="backward",
                           duration_ms=bwd_ms, version=self.current_version)
        else:
            loss_val, input_grad, _, _ = _do_last(input_data, labels)

        return loss_val, input_grad


    # ── Backward ──────────────────────────────────────────────────────────

    def train_backward_central(self, iter_id: int, grad: torch.Tensor):
        """Central backward — respects defer_step flag.
        
        MEMORY FIX: 
          - Checks defer_step before zero_grad/step (was missing before)
          - Uses set_to_none=True
          - Explicitly deletes intermediates and frees graph
        """
        assert iter_id in self.intermediates
        el = self.event_logger

        def _do_backward():
            grad_dev = grad.to(self.device)
            
            if not self.defer_step:
                # Recompute forward for a fresh autograd graph — the stored
                # output's graph is stale after earlier micro-batches'
                # optimizer.step() modified weight versions in-place.
                stored_input = self.intermediates[iter_id][0]
                if self.optimizer_ref is not None:
                    self.optimizer_ref.zero_grad(set_to_none=True)
                output = self.sub_model(stored_input)
                output.backward(gradient=grad_dev)
                if self.optimizer_ref is not None:
                    self.optimizer_ref.step()
                    self.current_version += 1
            else:
                # defer_step=True: stored output's autograd graph is still valid
                inter_output = self.intermediates[iter_id][1]
                inter_output.backward(gradient=grad_dev)
            
            # MEMORY FIX: aggressively free the intermediates and their autograd graphs
            del self.intermediates[iter_id]

        if el:
            with el.log_event(self.device_id, "backward", iter_id, phase="backward",
                              version=self.current_version):
                _do_backward()
        else:
            _do_backward()

    def train_backward_worker(self, iter_id: int, grad: torch.Tensor) -> torch.Tensor:
        """Non-last worker backward — respects defer_step flag.
        
        MEMORY FIX:
          - Checks defer_step before zero_grad/step (was missing before)
          - Uses set_to_none=True
          - Detaches input_grad to break graph reference
          - Explicitly deletes intermediates
        """
        assert iter_id in self.intermediates
        el = self.event_logger

        def _do_backward():
            grad_dev = grad.to(self.device)
            
            if not self.defer_step:
                # Recompute forward for a fresh autograd graph.
                stored_input = self.intermediates[iter_id][0].detach().requires_grad_(True)
                if self.optimizer_ref is not None:
                    self.optimizer_ref.zero_grad(set_to_none=True)
                output = self.sub_model(stored_input)
                output.backward(gradient=grad_dev)
                if self.optimizer_ref is not None:
                    self.optimizer_ref.step()
                    self.current_version += 1
                input_grad = stored_input.grad.detach().clone() if stored_input.grad is not None else torch.zeros_like(stored_input)
            else:
                # defer_step=True: stored output's autograd graph is still valid
                inter_input = self.intermediates[iter_id][0]
                inter_output = self.intermediates[iter_id][1]
                inter_output.backward(gradient=grad_dev)
                input_grad = inter_input.grad.detach().clone() if inter_input.grad is not None else torch.zeros_like(inter_input)
            
            # MEMORY FIX: aggressively free intermediates and their autograd graphs
            del self.intermediates[iter_id]
            
            return input_grad

        if el:
            with el.log_event(self.device_id, "backward", iter_id, phase="backward",
                              version=self.current_version):
                result = _do_backward()
        else:
            result = _do_backward()

        return result


logger.info("L2: ComputeBackend + ConfidantComputeBackend loaded (defer_step mode)")


In [ ]:
# =============================================================================
# L3: DATASET
# =============================================================================
# Abstract: DatasetBackend
# Concrete: ConfidantDatasetBackend
# Maps to: Dataset.java + C++ datasets
# =============================================================================


class DatasetBackend(ABC):
    @abstractmethod
    def create_dataset(self, name: str, path: str, batch_size: int, **kwargs): ...
    @abstractmethod
    def get_dataloader(self) -> DataLoader: ...
    @abstractmethod
    def get_data_len(self) -> int: ...
    @abstractmethod
    def reset(self): ...


class ConfidantDatasetBackend(DatasetBackend):
    """PyTorch dataset backend — maps to Dataset.java"""

    def __init__(self):
        self.dataset: Optional[TorchDataset] = None
        self.dataloader: Optional[DataLoader] = None
        self._data_len: int = 0
        self._batch_size: int = 16

    def create_dataset(self, name: str, path: str, batch_size: int, **kwargs):
        """Create dataset by name. For now supports dummy/synthetic data."""
        self._batch_size = batch_size

        if name == "dummy" or name == "synthetic":
            num_samples = kwargs.get('num_samples', 320)
            seq_len = kwargs.get('seq_len', 128)
            hidden_dim = kwargs.get('hidden_dim', 768)
            num_classes = kwargs.get('num_classes', 2)

            inputs = torch.randn(num_samples, seq_len, hidden_dim)
            labels = torch.randint(0, num_classes, (num_samples,))
            self.dataset = TensorDataset(inputs, labels)
        else:
            raise ValueError(f"Unknown dataset: {name}. Supported: dummy, synthetic")

        self.dataloader = DataLoader(
            self.dataset,
            batch_size=batch_size,
            shuffle=True,
            drop_last=True,
        )
        self._data_len = len(self.dataloader)
        logger.info(f"Dataset '{name}' created: {len(self.dataset)} samples, "
                     f"{self._data_len} batches (bs={batch_size})")

    def get_dataloader(self) -> DataLoader:
        assert self.dataloader is not None, "Dataset not created"
        return self.dataloader

    def get_data_len(self) -> int:
        return self._data_len

    def reset(self):
        """Re-create dataloader (resets iteration). Maps to dataLoader->reset() in C++."""
        if self.dataset is not None:
            self.dataloader = DataLoader(
                self.dataset,
                batch_size=self._batch_size,
                shuffle=True,
                drop_last=True,
            )


# =============================================================================
# L4: OPTIMIZER
# =============================================================================
# Abstract: OptimizerBackend
# Concrete: ConfidantOptimizerBackend
# Maps to: Optimizer.java + C++ optimizer.cpp
# =============================================================================


class OptimizerBackend(ABC):
    @abstractmethod
    def create_optimizer(self, model: nn.Module, lr: float, **kwargs) -> optim.Optimizer: ...
    @abstractmethod
    def step(self): ...
    @abstractmethod
    def zero_grad(self): ...
    @abstractmethod
    def get_learning_rate(self) -> float: ...
    @abstractmethod
    def set_learning_rate(self, lr: float): ...
    @abstractmethod
    def reset(self): ...
    @abstractmethod
    def get_latest_version(self) -> int: ...


class ConfidantOptimizerBackend(OptimizerBackend):
    """PyTorch optimizer backend — maps to Optimizer.java + OptimizerWeightVersion.
    Supports weight versioning for pipeline-parallel backward passes.
    """

    def __init__(self):
        self.optimizer: Optional[optim.Optimizer] = None
        self._lr: float = 1e-4
        self._version: int = 0

    def create_optimizer(self, model: nn.Module, lr: float, **kwargs) -> optim.Optimizer:
        weight_decay = kwargs.get('weight_decay', 0.01)
        self._lr = lr
        self.optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        logger.info(f"Optimizer created: AdamW, lr={lr}, wd={weight_decay}")
        return self.optimizer

    def step(self):
        if self.optimizer is not None:
            self.optimizer.step()
            self._version += 1

    def zero_grad(self):
        if self.optimizer is not None:
            self.optimizer.zero_grad()

    def get_learning_rate(self) -> float:
        return self._lr

    def set_learning_rate(self, lr: float):
        self._lr = lr
        if self.optimizer is not None:
            for pg in self.optimizer.param_groups:
                pg['lr'] = lr

    def reset(self):
        """Reset optimizer state (called per epoch). Maps to Optimizer.resetOptimizer()."""
        self._version = 0
        # Optionally reset optimizer state entirely
        # For now we just reset version counter

    def get_latest_version(self) -> int:
        return self._version


logger.info("L3: DatasetBackend + L4: OptimizerBackend loaded")

In [ ]:
# =============================================================================
# L5: PROFILING & SCHEDULING
# =============================================================================
# Abstract: ProfilerBackend + SchedulerBackend
# Concrete: ConfidantProfiler + ConfidantScheduler
# Maps to: OfflineProfiler.java + DynamicScheduler.java + profiler.cpp
# =============================================================================


class ProfilerBackend(ABC):
    @abstractmethod
    def profile_layer(self, model: nn.Module, input_data: torch.Tensor,
                      num_iterations: int) -> Tuple[float, float, float]: ...
    @abstractmethod
    def profile_bandwidth(self, src_device: torch.device, dst_device: torch.device,
                          data_size_mb: float) -> float: ...
    @abstractmethod
    def get_memory_info(self, device: torch.device) -> Tuple[float, float]: ...
    @abstractmethod
    def get_time_interval(self, device_id: int, start: int, end: int, phase: int) -> float: ...
    @abstractmethod
    def get_output_size(self, layer_idx: int) -> float: ...
    @abstractmethod
    def get_bandwidth(self, device_id: int) -> float: ...
    @abstractmethod
    def get_computing_capacity(self, device_id: int) -> float: ...
    @abstractmethod
    def get_available_memory(self, device_id: int) -> float: ...


class ConfidantProfiler(ProfilerBackend):
    """GPU profiler + offline data store — maps to OfflineProfiler.java + profiler.cpp.
    
    Two modes:
    1. Active profiling: profile_layer() measures real GPU times using CUDA events.
    2. Offline data: stores aggregated results for the scheduler to query.
    """

    def __init__(self, num_devices: int = 2):
        self.num_devices = num_devices

        # Per-layer profiles: List[{forward_ms, backward_ms, memory_mb}]
        self.layer_profiles: Dict[int, List[Dict[str, float]]] = {
            d: [] for d in range(num_devices)
        }

        # Aggregated time intervals: device_id → {(start, end, phase) → time_ms}
        self.time_intervals: Dict[int, Dict[Tuple[int, int, int], float]] = {
            d: {} for d in range(num_devices)
        }

        # Per-layer output sizes
        self.output_sizes: List[float] = []

        # Per-device bandwidth (MB/s)
        self.bandwidths: List[float] = [0.0] * num_devices

        # Per-device computing capacity ratio
        self.computing_capacities: List[float] = [1.0] * num_devices

        # Per-device available memory (GB)
        self.available_memory: List[float] = [0.0] * num_devices

    def profile_layer(self, model: nn.Module, input_data: torch.Tensor,
                    num_iterations: int = 10) -> Tuple[float, float, float]:
        """Profile a single layer with CUDA events. Returns (fwd_ms, bwd_ms, peak_mem_mb).
        Matches the profiling logic from profiler.cpp using MNN::Timer.
        """
        device = next(model.parameters()).device
        torch.cuda.set_device(device)  # <-- CRITICAL: ensure events go to correct GPU
        model.train()

        input_data = input_data.to(device)
        if not input_data.requires_grad:
            input_data = input_data.clone().detach().requires_grad_(True)

        # Warmup
        for _ in range(3):
            with torch.no_grad():
                _ = model(input_data)
        torch.cuda.synchronize(device)

        # Forward timing
        fwd_times = []
        for _ in range(num_iterations):
            torch.cuda.synchronize(device)
            s = torch.cuda.Event(enable_timing=True)
            e = torch.cuda.Event(enable_timing=True)
            s.record(torch.cuda.current_stream(device))
            with torch.no_grad():
                _ = model(input_data)
            e.record(torch.cuda.current_stream(device))
            torch.cuda.synchronize(device)
            fwd_times.append(s.elapsed_time(e))

        # Backward timing
        bwd_times = []
        for _ in range(num_iterations):
            torch.cuda.synchronize(device)
            model.zero_grad()
            if input_data.grad is not None:
                input_data.grad.zero_()
            out = model(input_data)
            grad_out = torch.ones_like(out)
            s = torch.cuda.Event(enable_timing=True)
            e = torch.cuda.Event(enable_timing=True)
            s.record(torch.cuda.current_stream(device))
            out.backward(gradient=grad_out)
            e.record(torch.cuda.current_stream(device))
            torch.cuda.synchronize(device)
            bwd_times.append(s.elapsed_time(e))

        # Memory
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        model.zero_grad()
        out = model(input_data)
        out.backward(gradient=torch.ones_like(out))
        peak_mem_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)

        fwd_ms = float(np.median(fwd_times[3:]))
        bwd_ms = float(np.median(bwd_times[3:]))

        model.zero_grad()
        torch.cuda.empty_cache()
        return fwd_ms, bwd_ms, peak_mem_mb

    def profile_bandwidth(self, src_device: torch.device, dst_device: torch.device,
                          data_size_mb: float = 100.0) -> float:
        """Measure P2P bandwidth between two GPUs in MB/s.
        Maps to profiler.profileBandwidth() in Java/C++.
        """
        num_bytes = int(data_size_mb * 1024 * 1024)
        src_tensor = torch.empty(num_bytes // 4, dtype=torch.float32, device=src_device)

        # Warmup
        for _ in range(3):
            _ = src_tensor.to(dst_device)
        torch.cuda.synchronize()

        times = []
        for _ in range(10):
            torch.cuda.synchronize()
            start = time.time()
            _ = src_tensor.to(dst_device)
            torch.cuda.synchronize()
            times.append(time.time() - start)

        avg_time = np.median(times[3:])
        bandwidth_mbs = data_size_mb / avg_time if avg_time > 0 else float('inf')
        return bandwidth_mbs

    def get_memory_info(self, device: torch.device) -> Tuple[float, float]:
        """Returns (available_gb, total_gb). Maps to MemoryInfo RPC."""
        if not torch.cuda.is_available():
            return (0.0, 0.0)
        props = torch.cuda.get_device_properties(device)
        total_gb = props.total_memory / (1024 ** 3)
        allocated_gb = torch.cuda.memory_allocated(device) / (1024 ** 3)
        return (total_gb - allocated_gb, total_gb)

    def build_time_intervals(self, device_id: int, profiles: List[Dict[str, float]]):
        """Build all (start, end, phase) intervals from per-layer profiles."""
        n = len(profiles)
        for s in range(n):
            for e in range(s, n):
                fwd = sum(profiles[i]['forward_ms'] for i in range(s, e + 1))
                bwd = sum(profiles[i]['backward_ms'] for i in range(s, e + 1))
                self.time_intervals[device_id][(s, e, 0)] = fwd
                self.time_intervals[device_id][(s, e, 1)] = bwd

    # --- Offline data getters (used by Scheduler) ---
    def get_time_interval(self, device_id: int, start: int, end: int, phase: int) -> float:
        return self.time_intervals.get(device_id, {}).get((start, end, phase), 0.0)

    def get_output_size(self, layer_idx: int) -> float:
        return self.output_sizes[layer_idx] if layer_idx < len(self.output_sizes) else 0.0

    def get_bandwidth(self, device_id: int) -> float:
        return self.bandwidths[device_id] if device_id < len(self.bandwidths) else 1.0

    def get_computing_capacity(self, device_id: int) -> float:
        return self.computing_capacities[device_id] if device_id < len(self.computing_capacities) else 1.0

    def get_available_memory(self, device_id: int) -> float:
        return self.available_memory[device_id] if device_id < len(self.available_memory) else 0.0


class SchedulerBackend(ABC):
    @abstractmethod
    def calculate_partition_point(self, is_average: bool) -> List[int]: ...
    @abstractmethod
    def calculate_partition_point_memory(self, is_average: bool) -> List[int]: ...


class ConfidantScheduler(SchedulerBackend):
    """DP-based dynamic scheduler — exact translation of DynamicScheduler.java"""

    def __init__(self, profiler: ProfilerBackend, num_layers: int, num_devices: int):
        self.profiler = profiler
        self.num_layers = num_layers
        self.num_devices = num_devices

    def _get_time(self, device_id: int, start: int, end: int) -> float:
        """Total forward+backward time for a layer range on a device."""
        fwd = self.profiler.get_time_interval(device_id, start, end, 0)
        bwd = self.profiler.get_time_interval(device_id, start, end, 1)
        return fwd + bwd

    def _get_comm_time(self, layer_idx: int, device_id: int) -> float:
        """Communication time for transferring output of layer_idx from device_id to next."""
        output_size = self.profiler.get_output_size(layer_idx)
        bandwidth = self.profiler.get_bandwidth(device_id)
        return output_size / bandwidth if bandwidth > 0 else 0.0

    def calculate_partition_point(self, is_average: bool = True) -> List[int]:
        """DP partition — exact translation of DynamicScheduler.java"""
        n = self.num_layers
        k = self.num_devices

        logger.info(f"Computing partition: {n} layers, {k} devices")

        if is_average:
            # Use average computing capacity
            capacities = [self.profiler.get_computing_capacity(d) for d in range(k)]
        else:
            capacities = [1.0] * k

        # DP: dp[i][j] = min bottleneck time to assign layers 0..i across devices 0..j
        INF = float('inf')
        dp = [[INF] * k for _ in range(n)]
        split = [[-1] * k for _ in range(n)]

        # Base case: all layers 0..i on device 0
        for i in range(n):
            dp[i][0] = self._get_time(0, 0, i) / capacities[0]

        # Fill DP table
        for j in range(1, k):
            for i in range(j, n):
                for m in range(j - 1, i):
                    # Layers 0..m on devices 0..j-1, layers m+1..i on device j
                    compute_time = self._get_time(j, m + 1, i) / capacities[j]
                    comm_time = self._get_comm_time(m, j - 1)
                    cost = max(dp[m][j - 1], compute_time + comm_time)

                    if cost < dp[i][j]:
                        dp[i][j] = cost
                        split[i][j] = m

        # Backtrack to find partition points
        points = []
        i, j = n - 1, k - 1
        while j > 0:
            m = split[i][j]
            points.append(m)
            i = m
            j -= 1
        points.reverse()

        bottleneck = dp[n - 1][k - 1]
        logger.info(f"Partition: {points}, Bottleneck: {bottleneck:.2f}ms")
        return points

    def calculate_partition_point_memory(self, is_average: bool = True) -> List[int]:
        """DP partition with memory constraint. Same as above but filters
        solutions that exceed available memory per device."""
        # For now, delegate to standard partition
        # TODO: Add memory-constrained DP variant
        return self.calculate_partition_point(is_average)


logger.info("L5: ProfilerBackend + SchedulerBackend loaded")

In [ ]:
# =============================================================================
# L6: COMMUNICATION
# =============================================================================
# Abstract: CommunicationManager
# Concrete: CommunicationManagerConfidant (direct calls + .to(device))
# Maps to: grpcAPI/ + grpcRequest/ + worker.proto + central.proto
#
# CRITICAL: Handler dispatch is ASYNCHRONOUS via ThreadPoolExecutor.
# The Java code uses async gRPC; without threading, the 1F1B commit-wait
# logic deadlocks because backward handlers are called inline during
# forward handlers in the same thread.
# =============================================================================


class CommunicationManager(ABC):
    """Abstract communication layer — all RPCs from worker.proto + central.proto.
    
    Implementations can use gRPC, torch.distributed.rpc, HTTP, or direct calls.
    Method names map 1:1 to proto service RPCs.
    """

    # --- Lifecycle ---
    @abstractmethod
    def start_server(self, role: str, device_id: int, address: str): ...
    @abstractmethod
    def stop_server(self): ...
    @abstractmethod
    def connect_to_peer(self, device_id: int, address: str): ...

    # --- Worker RPCs (from WorkerGreeter in worker.proto) ---
    @abstractmethod
    def send_forward(self, target_id: int, iter_id: int, model_idx: int,
                     version: int, lr: float, data: torch.Tensor): ...
    @abstractmethod
    def send_backward_to_worker(self, target_id: int, iter_id: int,
                                model_idx: int, grad: torch.Tensor): ...
    @abstractmethod
    def send_labels(self, target_id: int, iter_id: int, labels: torch.Tensor): ...
    @abstractmethod
    def send_start_epoch(self, target_id: int, epoch: int, lr: float, data_len: int): ...
    @abstractmethod
    def send_set_basic_info(self, target_id: int, info: Dict[str, Any]): ...
    @abstractmethod
    def send_update_workers(self, target_id: int, idx: int, workers: Dict[int, str]): ...
    @abstractmethod
    def is_available(self, target_id: int) -> bool: ...
    @abstractmethod
    def measure_bandwidth(self, target_id: int) -> float: ...
    @abstractmethod
    def get_memory_info(self, target_id: int) -> Tuple[float, float]: ...

    # --- Central RPCs (from CentralGreeter in central.proto) ---
    @abstractmethod
    def send_backward_to_central(self, iter_id: int, model_idx: int,
                                 grad: torch.Tensor): ...

    # --- Serialization helpers ---
    @abstractmethod
    def serialize_tensor(self, tensor: torch.Tensor) -> bytes: ...
    @abstractmethod
    def deserialize_tensor(self, data: bytes, shape: List[int],
                           device: torch.device) -> torch.Tensor: ...


class CommunicationManagerConfidant(CommunicationManager):
    """Communication using direct function calls with GPU tensor transfers.
    
    For single-node multi-GPU: calls handlers directly with .to(device) transfers.
    Handler dispatch is ASYNCHRONOUS (ThreadPoolExecutor) matching the Java
    gRPC async behavior. Without this, 1F1B commit-wait deadlocks in single-thread.
    
    For multi-node: swap this with a gRPC implementation — the abstract interface
    is identical, only the transport changes.
    
    Handler registration pattern matches the gRPC service registration in
    GrpcServer.java (registerServices for WorkerGreeter / CentralGreeter).
    """

    def __init__(self, event_logger: Optional[ConfidantEventLogger] = None,
                 max_workers: int = 8):
        self.event_logger = event_logger

        # Handler registrations: device_id → handler object
        self.forward_handlers: Dict[int, Callable] = {}
        self.backward_handlers: Dict[int, Callable] = {}
        self.label_handlers: Dict[int, Callable] = {}
        self.epoch_handlers: Dict[int, Callable] = {}
        self.basic_info_handlers: Dict[int, Callable] = {}
        self.central_backward_handler: Optional[Callable] = None

        # Device mapping: device_id → torch.device
        self.device_map: Dict[int, torch.device] = {}

        # Thread pool for async handler dispatch (matches Java gRPC async behavior)
        self._executor = ThreadPoolExecutor(
            max_workers=max_workers,
            thread_name_prefix="comm",
        )
        self._pending: List[Future] = []  # track in-flight futures
        self._lock = threading.Lock()       # protects _pending list

        # Error flag — set when any handler raises an exception.
        # Other threads check this to avoid waiting forever on a
        # notification that will never come (the handler that would
        # send it has crashed).
        self._error: Optional[Exception] = None

        self._started = False

    @property
    def has_error(self) -> bool:
        return self._error is not None

    def start_server(self, role: str, device_id: int, address: str):
        self.device_map[device_id] = torch.device(
            f"cuda:{device_id}" if torch.cuda.is_available() else "cpu"
        )
        self._started = True
        logger.info(f"CommManager: {role} device {device_id} started at {address}")

    def stop_server(self):
        self.wait_all()
        self._executor.shutdown(wait=True)
        self._started = False

    def connect_to_peer(self, device_id: int, address: str):
        """In single-node mode, just register the device mapping."""
        if device_id not in self.device_map:
            self.device_map[device_id] = torch.device(
                f"cuda:{device_id}" if torch.cuda.is_available() else "cpu"
            )

    def wait_all(self):
        """Wait for all in-flight async handler calls to complete,
        INCLUDING futures added by cascading handler dispatches.
        
        Handlers can trigger new handler submissions (e.g., forward handler
        on worker 1 → send_forward to worker 2 → new future). A simple
        snapshot-based wait would miss these cascading futures. Instead, we
        drain in a loop: wait for current batch, then check for new futures,
        repeat until no pending futures remain.
        
        This mirrors Java gRPC's behavior where the async stub ensures all
        cascading RPCs complete before the call returns.
        """
        while True:
            with self._lock:
                if not self._pending:
                    break
                # Take batch and clear — new futures from handlers will
                # be appended to the now-empty list during our wait
                batch = list(self._pending)
                self._pending.clear()

            for f in batch:
                try:
                    f.result()  # blocks until done; re-raises exceptions
                except Exception as e:
                    logger.error(f"CommManager: handler raised exception: {e}")
                    raise

    def _submit(self, fn: Callable, *args):
        """Submit a handler call to the thread pool.
        
        Wraps the handler to catch and log exceptions IMMEDIATELY
        (instead of silently storing them in the Future). This is critical:
        in the Java gRPC implementation, handler exceptions propagate via
        the RPC error mechanism. In Python ThreadPoolExecutor, exceptions
        are silently swallowed until Future.result() is called — but by then,
        downstream handlers are deadlocked waiting for a notification that
        will never come (because the crashed handler never incremented
        forwardId/backwardId and never called notify_all).
        """
        comm_ref = self  # closure for error flag

        def _safe_handler(*a):
            try:
                return fn(*a)
            except Exception as e:
                logger.error(f"CommManager handler CRASHED: {fn.__qualname__} → {e}", exc_info=True)
                comm_ref._error = e
                # Wake up ANY threads waiting on commit conditions so they
                # can detect the error and bail out instead of waiting forever.
                raise

        future = self._executor.submit(_safe_handler, *args)
        with self._lock:
            self._pending.append(future)
            # Prune completed futures periodically
            if len(self._pending) > 100:
                self._pending = [f for f in self._pending if not f.done()]

    # --- Registration (maps to gRPC service binding) ---
    def register_forward_handler(self, device_id: int, handler: Callable):
        self.forward_handlers[device_id] = handler

    def register_backward_handler(self, device_id: int, handler: Callable):
        self.backward_handlers[device_id] = handler

    def register_label_handler(self, device_id: int, handler: Callable):
        self.label_handlers[device_id] = handler

    def register_epoch_handler(self, device_id: int, handler: Callable):
        self.epoch_handlers[device_id] = handler

    def register_basic_info_handler(self, device_id: int, handler: Callable):
        self.basic_info_handlers[device_id] = handler

    def register_central_backward_handler(self, handler: Callable):
        self.central_backward_handler = handler

    # --- Worker RPCs ---
    def send_forward(self, target_id: int, iter_id: int, model_idx: int,
                     version: int, lr: float, data: torch.Tensor):
        """Maps to HandleForward RPC → TrainWorkerAPI.handleForward()
        
        Tensor transfer is SYNCHRONOUS (must complete before handler sees it).
        Handler dispatch is ASYNC (matches gRPC non-blocking stub).
        """
        handler = self.forward_handlers.get(target_id)
        assert handler is not None, f"No forward handler for device {target_id}"

        target_device = self.device_map.get(target_id, torch.device('cpu'))
        el = self.event_logger

        # Synchronous tensor transfer
        if el:
            with el.log_event(target_id, "comm_send_fwd", iter_id, phase="forward",
                              src_device=model_idx - 1, dst_device=target_id,
                              tensor_bytes=data.nelement() * data.element_size()):
                data_transferred = data.detach().to(target_device)
        else:
            data_transferred = data.detach().to(target_device)

        request = {
            'iterId': iter_id,
            'modelIdx': model_idx,
            'version': version,
            'lr': lr,
            'data': data_transferred,
        }

        # Async dispatch (like gRPC async stub)
        self._submit(handler, request)

    def send_backward_to_worker(self, target_id: int, iter_id: int,
                                model_idx: int, grad: torch.Tensor):
        """Maps to SendTrainBackward RPC → TrainWorkerAPI.sendTrainBackward()"""
        handler = self.backward_handlers.get(target_id)
        assert handler is not None, f"No backward handler for device {target_id}"

        target_device = self.device_map.get(target_id, torch.device('cpu'))
        el = self.event_logger

        if el:
            with el.log_event(target_id, "comm_send_bwd", iter_id, phase="backward",
                              dst_device=target_id,
                              tensor_bytes=grad.nelement() * grad.element_size()):
                grad_transferred = grad.detach().to(target_device)
        else:
            grad_transferred = grad.detach().to(target_device)

        data = {
            'iterId': iter_id,
            'modelIdx': model_idx,
            'data': grad_transferred,
        }

        # Async dispatch
        self._submit(handler, data)

    def send_labels(self, target_id: int, iter_id: int, labels: torch.Tensor):
        """Maps to Labels RPC — synchronous (must arrive before forward handler)."""
        handler = self.label_handlers.get(target_id)
        assert handler is not None, f"No label handler for device {target_id}"
        handler(iter_id, labels)  # sync: labels must be cached before forward runs

    def send_start_epoch(self, target_id: int, epoch: int, lr: float, data_len: int):
        """Maps to StartEpoch RPC — synchronous (setup must complete before training)."""
        handler = self.epoch_handlers.get(target_id)
        assert handler is not None, f"No epoch handler for device {target_id}"
        handler(epoch, lr, data_len)

    def send_set_basic_info(self, target_id: int, info: Dict[str, Any]):
        """Maps to SetBasicInfo RPC"""
        handler = self.basic_info_handlers.get(target_id)
        assert handler is not None, f"No basic_info handler for device {target_id}"
        handler(info)

    def send_update_workers(self, target_id: int, idx: int, workers: Dict[int, str]):
        pass  # Used in fault tolerance

    def is_available(self, target_id: int) -> bool:
        return target_id in self.forward_handlers

    def measure_bandwidth(self, target_id: int) -> float:
        return 0.0  # Use profiler instead

    def get_memory_info(self, target_id: int) -> Tuple[float, float]:
        device = self.device_map.get(target_id)
        if device and torch.cuda.is_available():
            props = torch.cuda.get_device_properties(device)
            total = props.total_memory / (1024 ** 3)
            used = torch.cuda.memory_allocated(device) / (1024 ** 3)
            return (total - used, total)
        return (0.0, 0.0)

    # --- Central RPCs ---
    def send_backward_to_central(self, iter_id: int, model_idx: int,
                                 grad: torch.Tensor):
        """Maps to SendTrainBackward RPC → TrainCentralAPI
        
        Async dispatch — allows worker thread to return immediately,
        preventing deadlock in commit-wait chain.
        """
        assert self.central_backward_handler is not None
        central_device = self.device_map.get(0, torch.device('cpu'))
        el = self.event_logger

        if el:
            with el.log_event(0, "comm_send_bwd", iter_id, phase="backward",
                              dst_device=0,
                              tensor_bytes=grad.nelement() * grad.element_size()):
                grad_transferred = grad.detach().to(central_device)
        else:
            grad_transferred = grad.detach().to(central_device)

        data = {
            'iterId': iter_id,
            'modelIdx': 0,
            'data': grad_transferred,
        }

        # Async dispatch
        self._submit(self.central_backward_handler, data)

    # --- Serialization ---
    def serialize_tensor(self, tensor: torch.Tensor) -> bytes:
        """Serialize tensor to bytes (for network transport). Maps to GrpcUtils."""
        buf = torch.ByteStorage.from_buffer(tensor.cpu().numpy().tobytes())
        return bytes(buf)

    def deserialize_tensor(self, data: bytes, shape: List[int],
                           device: torch.device) -> torch.Tensor:
        arr = np.frombuffer(data, dtype=np.float32).reshape(shape)
        return torch.from_numpy(arr.copy()).to(device)


logger.info("L6: CommunicationManager + CommunicationManagerConfidant loaded (async dispatch)")

In [ ]:
# =============================================================================
# L7: FAULT TOLERANCE
# =============================================================================
# Abstract: FaultToleranceBackend
# Concrete: ConfidantFaultTolerance
# Maps to: faultTolerance/ (PassiveFTHandler, ProactiveFTHandler, ReplicationUtils)
# =============================================================================


class FaultToleranceBackend(ABC):
    """Abstract fault tolerance layer."""

    @abstractmethod
    def detect_failure(self, iter_id: int, timeout_ms: int) -> bool: ...
    @abstractmethod
    def handle_passive_timeout(self, iter_id: int): ...
    @abstractmethod
    def replicate_weights(self, replication_type: str, compute_backends: Dict[int, ComputeBackend]): ...
    @abstractmethod
    def redistribute_weights(self, failed_devices: List[int],
                             new_partition: List[int],
                             compute_backends: Dict[int, ComputeBackend]): ...
    @abstractmethod
    def restart_sync_state(self, device_idx: int, workers: Dict[int, str],
                           partition: List[int]): ...
    @abstractmethod
    def commit_fault_sync(self, iter_id: int, partition: List[int]): ...
    @abstractmethod
    def save_checkpoint(self, epoch: int, iter_id: int,
                        compute_backends: Dict[int, ComputeBackend],
                        optimizer_backends: Dict[int, OptimizerBackend]): ...
    @abstractmethod
    def load_checkpoint(self, path: str) -> Dict[str, Any]: ...


class ConfidantFaultTolerance(FaultToleranceBackend):
    """Confidant fault tolerance — passive + proactive strategies.
    
    Passive FT (from PassiveFTHandler.java):
      1. Backward timeout detection
      2. Weight redistribution across surviving devices
      3. Resume training from last good iteration
    
    Proactive FT (from ProactiveFTHandler.java):
      1. Device signals intent to exit
      2. Find substitute device
      3. Transfer weights to substitute
      4. Update worker registry
    
    Weight replication (from ReplicationUtils.java):
      - Local: replicate to adjacent device
      - Global: replicate to all devices
    """

    def __init__(self, state: StateManager, checkpoint_dir: str = "./checkpoints"):
        self.state = state
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

        # Replication storage: device_id → {layer_name: tensor}
        self.local_replicas: Dict[int, Dict[str, torch.Tensor]] = {}
        self.global_replicas: Dict[int, Dict[str, torch.Tensor]] = {}

    def detect_failure(self, iter_id: int, timeout_ms: int) -> bool:
        """Check if backward was received within timeout.
        Maps to setPassiveTimeout() in TrainCentral.java.
        """
        received = self.state.get_received_iter_ids()
        if iter_id not in received and self.state.get_system_status() == "NORMAL":
            logger.warning(f"FT: Backward not received for iter {iter_id} within {timeout_ms}ms")
            return True
        return False

    def handle_passive_timeout(self, iter_id: int):
        """3-phase passive FT recovery — maps to PassiveFTHandler.backwardTimeoutHandler().
        Phase 1: Mark system as RECOVERING
        Phase 2: Redistribute weights (re-partition excluding failed device)
        Phase 3: Resume from last good iteration
        """
        logger.warning(f"FT: Passive timeout triggered for iter {iter_id}")
        self.state.set_system_status("RECOVERING")

        # In full implementation:
        # 1. Identify failed device from timeout
        # 2. Call redistribute_weights() with new partition
        # 3. Call restart_sync_state() on all surviving workers
        # 4. Set start_iter_id and resume

        self.state.set_system_status("NORMAL")
        logger.info(f"FT: Recovery complete, resuming from iter {iter_id}")

    def replicate_weights(self, replication_type: str,
                          compute_backends: Dict[int, ComputeBackend]):
        """Replicate weights to other devices — maps to ReplicationUtils.replicateWeights().
        
        LOCAL_REPLICATION: save to adjacent device
        GLOBAL_REPLICATION: save to all devices
        """
        for device_id, backend in compute_backends.items():
            if backend.sub_model is not None:
                state_dict = {k: v.cpu().clone() for k, v in backend.sub_model.state_dict().items()}

                if replication_type == "LOCAL":
                    self.local_replicas[device_id] = state_dict
                elif replication_type == "GLOBAL":
                    self.global_replicas[device_id] = state_dict

        logger.info(f"FT: {replication_type} replication complete for {len(compute_backends)} devices")

    def redistribute_weights(self, failed_devices: List[int],
                             new_partition: List[int],
                             compute_backends: Dict[int, ComputeBackend]):
        """Redistribute weights after re-partitioning.
        Maps to RedistributionUtils in Java.
        """
        logger.info(f"FT: Redistributing weights. Failed: {failed_devices}, New partition: {new_partition}")
        # In full implementation:
        # 1. Collect all weight tensors from surviving devices
        # 2. Reassign layers according to new partition
        # 3. Load weights into new sub-models

    def restart_sync_state(self, device_idx: int, workers: Dict[int, str],
                           partition: List[int]):
        """Sync state after restart — maps to RestartSyncState RPC."""
        self.state.set_partition_point(partition)
        self.state.set_workers(workers)
        logger.info(f"FT: State synced for device {device_idx}")

    def commit_fault_sync(self, iter_id: int, partition: List[int]):
        """Commit fault sync — maps to CommitFaultSync RPC."""
        self.state.set_partition_point(partition)
        logger.info(f"FT: Fault sync committed at iter {iter_id}")

    def save_checkpoint(self, epoch: int, iter_id: int,
                        compute_backends: Dict[int, ComputeBackend],
                        optimizer_backends: Dict[int, OptimizerBackend]):
        """Save checkpoint for all devices."""
        ckpt = {
            'epoch': epoch,
            'iter_id': iter_id,
            'partition_points': self.state.get_partition_point(),
            'models': {},
            'optimizers': {},
        }
        for did, cb in compute_backends.items():
            if cb.sub_model is not None:
                ckpt['models'][did] = cb.sub_model.state_dict()
        for did, ob in optimizer_backends.items():
            if ob.optimizer is not None:
                ckpt['optimizers'][did] = ob.optimizer.state_dict()

        path = self.checkpoint_dir / f"ckpt_epoch{epoch}_iter{iter_id}.pt"
        torch.save(ckpt, path)
        logger.info(f"FT: Checkpoint saved: {path}")

    def load_checkpoint(self, path: str) -> Dict[str, Any]:
        """Load checkpoint."""
        ckpt = torch.load(path, map_location='cpu')
        logger.info(f"FT: Checkpoint loaded: {path}")
        return ckpt


logger.info("L7: FaultToleranceBackend + ConfidantFaultTolerance loaded")

In [ ]:
# =============================================================================
# L8: ORCHESTRATION
# =============================================================================
# Abstract: Orchestrator
# Concrete: ConfidantOrchestrator (CentralRole + WorkerRole)
# Maps to: TrainCentral.java + TrainWorker.java
#
# Architecture:
#   - Each device has its OWN ConfidantStateManager (commit dict + lock + condition)
#     matching Java where each device process has its own Training.java / CommonStates.
#   - The commit (forwardId, backwardId) tracks per-device 1F1B progress.
#   - Cross-device synchronization is handled by the communication layer.
#   - Central's semaphore lives on central's state (controls pipeline depth).
#
# Gradient accumulation:
#   PyTorch autograd requires weights to be unchanged between a batch's forward
#   and backward. The 1F1B schedule allows batch_diff outstanding forwards, so
#   optimizer.step() in backward(N) would modify weights that backward(N+1) needs.
#   Fix: defer optimizer.step() to epoch end (gradient accumulation).
#
# Safety mechanisms (Python-specific, not in Java):
#   - condition.wait(timeout=2.0) instead of unbounded wait
#   - comm.has_error check in wait loops to bail out if a handler crashed
#   These are needed because Python ThreadPoolExecutor silently swallows
#   handler exceptions, unlike Java gRPC which propagates errors via RPC status.
# =============================================================================


class Orchestrator(ABC):
    """Abstract orchestrator — top-level training control."""

    @abstractmethod
    def setup(self, full_model: nn.Module, **kwargs): ...
    @abstractmethod
    def start_train(self, num_epochs: int): ...


class CentralRole:
    """Central node role — exact translation of TrainCentral.java.
    
    The central owns device 0, runs the first pipeline stage,
    orchestrates epoch starts, enforces 1F1B scheduling.
    """

    def __init__(self, state: StateManager, compute: ComputeBackend,
                 optimizer: OptimizerBackend, dataset: DatasetBackend,
                 comm: CommunicationManager, ft: FaultToleranceBackend,
                 event_logger: Optional[ConfidantEventLogger] = None,
                 log_interval: int = 5,
                 epoch_end_callback: Optional[Callable] = None):
        self.state = state
        self.compute = compute
        self.optimizer = optimizer
        self.dataset = dataset
        self.comm = comm
        self.ft = ft
        self.el = event_logger
        self.log_interval = log_interval
        self.device_id = 0  # Central is always device 0
        self._epoch_end_callback = epoch_end_callback

    def start_train(self, num_epochs: int):
        """Maps to TrainCentral.startTrain() + TrainDistribute()"""
        logger.info("Central: Starting distributed training")
        data_len = self.dataset.get_data_len()

        for epoch in range(num_epochs):
            lr = self.optimizer.get_learning_rate()

            # Notify workers of new epoch (sendStartEpoch RPC — synchronous)
            workers = self.state.get_workers()
            for idx, url in workers.items():
                if idx != 0:
                    self.comm.send_start_epoch(idx, epoch, lr, data_len)

            self.optimizer.reset()

            if epoch > 0:
                self.state.set_start_iter_id(0)
            self.state.clear_received_iter_ids()

            # Central init_commit — sets forwardId=-1, backwardId=-1
            self.state.init_commit(data_len, epoch)
            epoch_start = time.time()
            self.state.set_start_time(epoch_start)

            # Tell event logger where this epoch starts
            if self.el:
                self.el.set_epoch_start(epoch_start)
                if self.el._dashboard:
                    self.el._dashboard.update_epoch(epoch)

            logger.info(f"\n{'='*60}")
            logger.info(f"Epoch {epoch + 1}/{num_epochs}  ({data_len} batches)")
            logger.info(f"{'='*60}")

            self._train_epoch_distribute(epoch)

            # Gradient accumulation (defer_step=True): step all optimizers
            # now that gradients have accumulated across all micro-batches.
            # When defer_step=False each micro-batch already stepped.
            if self.compute.defer_step:
                self.compute.step_optimizer()
                if self._epoch_end_callback:
                    self._epoch_end_callback(epoch)
                logger.info(f"  All optimizers stepped (epoch {epoch + 1})")

            # Log epoch duration
            epoch_ms = (time.time() - epoch_start) * 1000
            if self.el:
                self.el.record_event(0, "epoch_end", -1, "epoch",
                                     epoch_ms, timestamp=epoch_start,
                                     epoch=epoch, num_batches=data_len)

            logger.info(f"Epoch {epoch + 1} complete in {epoch_ms:.0f}ms")

    def _train_epoch_distribute(self, epoch: int):
        """Maps to TrainCentral.trainEpochDistribute()
        
        GRADIENT ACCUMULATION (defer_step=True):
          - Zero all grads at epoch start (central stage)
          - All micro-batches accumulate gradients without stepping
          - After all micro-batches complete, step is called by the orchestrator
        """
        self.compute.init_train_epoch()
        # Zero central's gradients at epoch start
        self.compute.zero_all_grads()
        
        data_len = self.state.get_commit()['dataLen']
        dataloader = self.dataset.get_dataloader()

        for iter_id, batch in enumerate(dataloader):
            if iter_id >= data_len:
                break

            # Check for handler errors before proceeding
            if hasattr(self.comm, 'has_error') and self.comm.has_error:
                raise RuntimeError(
                    f"Training aborted: handler exception detected — {self.comm._error}")

            sem = self.state.get_semaphore()
            sem.acquire()

            while self.state.get_system_status() != "NORMAL":
                time.sleep(1.0)

            self._train_one_batch_distribute(epoch, iter_id, batch)

        # Wait for all in-flight async operations to complete
        self.comm.wait_all()

    def _train_one_batch_distribute(self, epoch: int, iter_id: int, batch):
        """Maps to TrainCentral.trainOneBatchDistribute()
        
        1F1B enforcement via commit lock, exactly as in Java:
          - forwardId == iter_id - 1  (sequential forward ordering)
          - warmup phase: iter_id - startIterId - batchDiff < 0
          - steady state: backwardId == iter_id - batchDiff
        
        batchDiff = workerNum (total pipeline stages) for central.
        """
        inputs, labels = batch
        commit = self.state.get_commit()
        condition = self.state.get_commit_condition()

        # Weights replication check
        local_rep = self.state.get_local_replication_interval()
        if local_rep > 0 and iter_id > 0 and iter_id % local_rep == 0:
            self.ft.replicate_weights("LOCAL", {0: self.compute})

        # ── 1F1B Commit Wait (forward) ──
        # Java: TrainCentral.trainOneBatchDistribute() commitLock + condition.await()
        wait_start = time.time()
        with condition:
            batch_diff = self.state.get_worker_num()
            start_iter = self.state.get_start_iter_id()

            while not (
                commit['forwardId'] == iter_id - 1
                and (
                    iter_id - start_iter - batch_diff < 0
                    or commit['backwardId'] == iter_id - batch_diff
                )
            ):
                # Python safety: timeout + error check (Java uses gRPC error propagation)
                if hasattr(self.comm, 'has_error') and self.comm.has_error:
                    raise RuntimeError(f"Handler crashed: {self.comm._error}")
                condition.wait(timeout=2.0)
        wait_ms = (time.time() - wait_start) * 1000
        if self.el and wait_ms > 0.01:
            self.el.record_event(0, "commit_wait_fwd", iter_id, "forward", wait_ms)

        # ── Forward ── (central is always device 0)
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        inputs_gpu = inputs.to(device)
        output, _ = self.compute.train_forward(iter_id, inputs_gpu)

        if iter_id % self.log_interval == 0:
            logger.info(f"  Batch {iter_id} forward on central complete")

        # Update forwardId on CENTRAL's commit
        with condition:
            commit['forwardId'] += 1
            condition.notify_all()

        # Send to next worker (async)
        self._send_forward_intermediate(epoch, iter_id, output, labels)

    def _send_forward_intermediate(self, epoch: int, iter_id: int,
                                   output: torch.Tensor, labels: torch.Tensor):
        """Maps to TrainCentral.sendForwardIntermediate()"""
        next_idx = 1  # Central is device 0, next is always 1
        last_idx = self.state.get_worker_num() - 1

        # Send labels to last worker
        self.comm.send_labels(last_idx, iter_id, labels)

        # Send forward activation to next worker
        model_version = self.compute.current_version
        lr = self.optimizer.get_learning_rate()
        self.comm.send_forward(next_idx, iter_id, next_idx, model_version, lr, output)

    def handle_backward_intermediate(self, data: Dict[str, Any]):
        """Maps to TrainCentral.handleBackwardIntermediate(Map<>)
        
        Runs in a ThreadPool thread (async dispatch from comm layer).
        Accesses CENTRAL's commit (separate from worker commits).
        
        Java commit-wait condition:
          - backwardId == iter_id - 1  (sequential backward ordering)
          - forwardId == iter_id + batchDiff - 1  (enough forwards completed)
          - OR forwardId == dataLen - 1  (epoch drain phase — all forwards done)
        
        batchDiff = workerNum for central.
        """
        commit = self.state.get_commit()
        condition = self.state.get_commit_condition()
        iter_id = data['iterId']
        output_grad = data['data']

        # ── 1F1B Commit Wait (backward) ──
        wait_start = time.time()
        with condition:
            batch_diff = self.state.get_worker_num()

            while not (
                commit['backwardId'] == iter_id - 1
                and (
                    commit['forwardId'] == iter_id + batch_diff - 1
                    or commit['forwardId'] == commit['dataLen'] - 1
                )
            ):
                if hasattr(self.comm, 'has_error') and self.comm.has_error:
                    raise RuntimeError(f"Handler crashed: {self.comm._error}")
                condition.wait(timeout=2.0)
        wait_ms = (time.time() - wait_start) * 1000
        if self.el and wait_ms > 0.01:
            self.el.record_event(0, "commit_wait_bwd", iter_id, "backward", wait_ms)

        self.state.add_received_iter_id(iter_id)

        self.compute.train_backward_central(iter_id, output_grad)

        if iter_id % self.log_interval == 0:
            logger.info(f"  Batch {iter_id} backward on central complete")

        with condition:
            commit['backwardId'] += 1
            condition.notify_all()

        self.state.get_semaphore().release()


class WorkerRole:
    """Worker node role — exact translation of TrainWorker.java.
    
    Each worker has its OWN state (separate commit tracking).
    """

    def __init__(self, device_id: int, state: StateManager, compute: ComputeBackend,
                 optimizer: OptimizerBackend, comm: CommunicationManager,
                 ft: FaultToleranceBackend,
                 event_logger: Optional[ConfidantEventLogger] = None,
                 log_interval: int = 5):
        self.device_id = device_id
        self.state = state
        self.compute = compute
        self.optimizer = optimizer
        self.comm = comm
        self.ft = ft
        self.el = event_logger
        self.log_interval = log_interval

        # Labels cache (last worker only)
        self.labels_cache: Dict[int, torch.Tensor] = {}

    def set_basic_info(self, info: Dict[str, Any]):
        """Maps to TrainWorker.setBasicInfoHandler()"""
        partition = info.get('partition_points', [])
        self.state.set_model_name(info.get('model_name', ''))
        self.state.set_model_args(info.get('model_args', {}))
        self.state.set_partition_point(partition)
        logger.info(f"Worker {self.device_id}: received partition {partition}")

    def init_epoch(self, epoch: int, lr: float, data_len: int):
        """Maps to TrainWorker.initEpoch()
        
        Worker init_commit resets its OWN commit (separate from central's).
        """
        self.optimizer.set_learning_rate(lr)
        self.optimizer.reset()
        # Worker init_commit — resets worker's forwardId=-1, backwardId=-1
        self.state.init_commit(data_len, epoch)
        self.compute.init_train_epoch()
        # MEMORY FIX: zero grads at epoch start for gradient accumulation mode
        self.compute.zero_all_grads()
        self.labels_cache.clear()
        logger.debug(f"Worker {self.device_id}: epoch {epoch} initialized")

    def receive_labels(self, iter_id: int, labels: torch.Tensor):
        self.labels_cache[iter_id] = labels

    def handle_forward(self, data: Dict[str, Any]):
        """Maps to TrainWorker.handleForward() — handles both intermediate and last worker.
        
        Runs in a ThreadPool thread (async dispatch from comm layer).
        Accesses THIS WORKER's commit (separate from central's).
        
        Java commit-wait condition (forward):
          - forwardId == iter_id - 1  (sequential forward ordering)
          - warmup: iter_id - startIterId - batchDiff < 0
          - steady: backwardId == iter_id - batchDiff
        
        batchDiff = workerNum - curIdx (stages from this worker to pipeline end).
        """
        input_data = data['data']
        cur_idx = data['modelIdx']
        iter_id = data['iterId']

        commit = self.state.get_commit()
        condition = self.state.get_commit_condition()

        # Weight replication checks
        local_rep = self.state.get_local_replication_interval()
        if local_rep > 0 and iter_id > 0 and iter_id % local_rep == 0:
            self.ft.replicate_weights("LOCAL", {self.device_id: self.compute})

        global_rep = self.state.get_global_replication_interval()
        if global_rep > 0 and iter_id > 0 and iter_id % global_rep == 0:
            self.ft.replicate_weights("GLOBAL", {self.device_id: self.compute})

        # ── 1F1B Commit Wait (forward) ──
        wait_start = time.time()
        with condition:
            batch_diff = self.state.get_worker_num() - cur_idx
            start_iter = self.state.get_start_iter_id()

            while not (
                commit['forwardId'] == iter_id - 1
                and (
                    iter_id - start_iter - batch_diff < 0
                    or commit['backwardId'] == iter_id - batch_diff
                )
            ):
                if hasattr(self.comm, 'has_error') and self.comm.has_error:
                    raise RuntimeError(f"Handler crashed: {self.comm._error}")
                condition.wait(timeout=2.0)
        wait_ms = (time.time() - wait_start) * 1000
        if self.el and wait_ms > 0.01:
            self.el.record_event(self.device_id, "commit_wait_fwd", iter_id,
                                 "forward", wait_ms)

        worker_num = self.state.get_worker_num()

        if cur_idx != worker_num - 1:
            # -------- NOT last worker --------
            start_time = time.time()
            output = self.compute.train_intermediate(iter_id, input_data)
            elapsed = (time.time() - start_time) * 1000
            self.state.update_forward_time(elapsed)

            with condition:
                commit['forwardId'] += 1
                condition.notify_all()

            # Send to next worker
            next_idx = self.device_id + 1
            lr = self.optimizer.get_learning_rate()
            self.comm.send_forward(next_idx, iter_id, next_idx,
                                   self.compute.current_version, lr, output)

        else:
            # -------- LAST worker: forward + loss + backward --------
            labels = self.labels_cache.pop(iter_id, None)
            assert labels is not None, f"Worker {self.device_id}: no labels for iter {iter_id}"

            start_time = time.time()
            loss_val, input_grad = self.compute.train_intermediate_last(iter_id, input_data, labels)
            elapsed = (time.time() - start_time) * 1000

            # Push loss to live dashboard
            if self.el and self.el._dashboard:
                self.el._dashboard.update_loss(iter_id, loss_val)

            if iter_id % self.log_interval == 0:
                logger.info(f"  Batch {iter_id}: loss={loss_val:.4f} ({elapsed:.1f}ms)")

            with condition:
                commit['forwardId'] += 1
                commit['backwardId'] += 1
                condition.notify_all()

            # Send gradient backward
            self._send_backward(cur_idx - 1, iter_id, input_grad)

    def handle_backward(self, data: Dict[str, Any]):
        """Maps to TrainWorker.handleBackward()
        
        Java commit-wait condition (backward):
          - backwardId == iter_id - 1  (sequential backward ordering)
          - forwardId == iter_id + batchDiff - 1  (enough forwards completed)
          - OR forwardId == dataLen - 1  (epoch drain phase)
        
        batchDiff = workerNum - curIdx.
        """
        cur_idx = data['modelIdx']
        iter_id = data['iterId']
        grad = data['data']

        commit = self.state.get_commit()
        condition = self.state.get_commit_condition()

        # ── 1F1B Commit Wait (backward) ──
        wait_start = time.time()
        with condition:
            batch_diff = self.state.get_worker_num() - cur_idx

            while not (
                commit['backwardId'] == iter_id - 1
                and (
                    commit['forwardId'] == iter_id + batch_diff - 1
                    or commit['forwardId'] == commit['dataLen'] - 1
                )
            ):
                if hasattr(self.comm, 'has_error') and self.comm.has_error:
                    raise RuntimeError(f"Handler crashed: {self.comm._error}")
                condition.wait(timeout=2.0)
        wait_ms = (time.time() - wait_start) * 1000
        if self.el and wait_ms > 0.01:
            self.el.record_event(self.device_id, "commit_wait_bwd", iter_id,
                                 "backward", wait_ms)

        start_time = time.time()
        input_grad = self.compute.train_backward_worker(iter_id, grad)
        elapsed = (time.time() - start_time) * 1000
        self.state.update_backward_time(elapsed)

        if iter_id % self.log_interval == 0:
            logger.info(f"  Batch {iter_id} backward on worker {self.device_id} ({elapsed:.1f}ms)")

        with condition:
            commit['backwardId'] += 1
            condition.notify_all()

        self._send_backward(cur_idx - 1, iter_id, input_grad)

    def _send_backward(self, target_idx: int, iter_id: int, grad: torch.Tensor):
        """Maps to TrainWorker.sendBackward()"""
        if target_idx == 0:
            self.comm.send_backward_to_central(iter_id, 0, grad)
        else:
            self.comm.send_backward_to_worker(target_idx, iter_id, target_idx, grad)


class ConfidantOrchestrator(Orchestrator):
    """Top-level orchestrator — wires all layers together.
    
    Architecture decisions:
      - SEPARATE state per device (each has own commit tracking for 1F1B)
      - Gradient accumulation (defer_step=True): zero_grad at epoch start,
        optimizer.step at epoch end. Prevents autograd weight-versioning errors.
      - ThreadPoolExecutor in comm layer for async handler dispatch.
    """

    def __init__(self, num_devices: int = 2, num_epochs: int = 3,
                 batch_size: int = 16, lr: float = 1e-4,
                 checkpoint_dir: str = "./checkpoints",
                 event_logger: Optional[ConfidantEventLogger] = None,
                 enable_dashboard: bool = True):
        self.num_devices = num_devices
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.lr = lr
        self.checkpoint_dir = checkpoint_dir
        self._enable_dashboard = enable_dashboard

        # Event logger — passed to all instrumented layers
        self.event_logger = event_logger or EVENT_LOGGER

        # Create shared communication layer (with logger)
        # Scale thread pool with device count — each device can have
        # batch_diff concurrent blocked handlers (fwd + bwd), so we need
        # enough threads to avoid starvation. Java gRPC uses unbounded pools.
        self.comm = CommunicationManagerConfidant(
            event_logger=self.event_logger,
            max_workers=num_devices * 32,
        )

        # Per-device layers
        self.states: Dict[int, ConfidantStateManager] = {}
        self.computes: Dict[int, ConfidantComputeBackend] = {}
        self.optimizers: Dict[int, ConfidantOptimizerBackend] = {}
        self.fts: Dict[int, ConfidantFaultTolerance] = {}

        # Roles
        self.central: Optional[CentralRole] = None
        self.workers: Dict[int, WorkerRole] = {}

        # Dataset (shared, on central)
        self.dataset = ConfidantDatasetBackend()

        # Scheduler
        self.profiler: Optional[ConfidantProfiler] = None
        self.scheduler: Optional[ConfidantScheduler] = None

    def setup(self, full_model: nn.Module, dataset_name: str = "dummy",
              dataset_path: str = "", profile: bool = True, **dataset_kwargs):
        """Setup the entire system — create all layers, partition model, wire handlers."""

        num_layers = len(full_model.layers)
        logger.info(f"\n{'='*60}")
        logger.info(f"CONFIDANT SETUP: {self.num_devices} devices, {num_layers} layers")
        logger.info(f"{'='*60}")

        # Infer hidden dimension from model or dataset kwargs
        hidden_dim = dataset_kwargs.get('hidden_dim', None)
        if hidden_dim is None:
            # Try to infer from model structure
            first_layer = full_model.layers[0]
            if hasattr(first_layer, 'in_features'):
                hidden_dim = first_layer.in_features
            elif isinstance(first_layer, nn.Sequential) and len(first_layer) > 0:
                child = first_layer[0]
                if hasattr(child, 'in_features'):
                    hidden_dim = child.in_features
                elif hasattr(child, 'hidden_size'):
                    hidden_dim = child.hidden_size
                else:
                    # Walk all parameters to find first linear layer
                    for m in first_layer.modules():
                        if isinstance(m, nn.Linear):
                            hidden_dim = m.in_features
                            break
                        elif isinstance(m, nn.LayerNorm):
                            hidden_dim = m.normalized_shape[0]
                            break
            if hidden_dim is None:
                hidden_dim = 768  # fallback
                logger.warning(f"Could not infer hidden_dim, using default {hidden_dim}")
        logger.info(f"  Hidden dimension: {hidden_dim}")

        # --- Step 1: Profiling & Scheduling ---
        logger.info("\n[1/5] Profiling & Scheduling...")
        self.profiler = ConfidantProfiler(self.num_devices)

        if profile and torch.cuda.is_available():
            for device_id in range(self.num_devices):
                device = torch.device(f"cuda:{device_id}")
                
                torch.cuda.set_device(device)
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats(device)
                torch.cuda.synchronize(device)
                
                local_warmup = torch.randn(self.batch_size, 128,
                    hidden_dim, device=device,
                    requires_grad=True)
                warmup_layer = copy.deepcopy(full_model.layers[0]).to(device)
                for _ in range(5):
                    out = warmup_layer(local_warmup)
                    out.backward(torch.ones_like(out))
                del warmup_layer, local_warmup
                torch.cuda.empty_cache()
                torch.cuda.synchronize(device)

                profiles = []
                for layer_idx, layer in enumerate(full_model.layers):
                    dummy = torch.randn(
                        self.batch_size, 128,
                        hidden_dim,
                        device=device, requires_grad=True
                    )
                    fwd, bwd, mem = self.profiler.profile_layer(
                        copy.deepcopy(layer).to(device), dummy, num_iterations=5
                    )
                    profiles.append({'forward_ms': fwd, 'backward_ms': bwd, 'memory_mb': mem})
                    logger.info(f"  GPU {device_id} Layer {layer_idx}: "
                                f"fwd={fwd:.2f}ms bwd={bwd:.2f}ms mem={mem:.0f}MB")

                self.profiler.layer_profiles[device_id] = profiles
                self.profiler.build_time_intervals(device_id, profiles)

            if self.num_devices > 1:
                bw = self.profiler.profile_bandwidth(
                    torch.device('cuda:0'), torch.device('cuda:1'))
                for d in range(self.num_devices):
                    self.profiler.bandwidths[d] = bw
                logger.info(f"  Bandwidth: {bw:.0f} MB/s")

            for d in range(self.num_devices):
                avail, total = self.profiler.get_memory_info(torch.device(f'cuda:{d}'))
                self.profiler.available_memory[d] = avail

            self.profiler.output_sizes = [float(hidden_dim)] * num_layers
        else:
            logger.info("  Skipping GPU profiling (no CUDA or profile=False)")
            self.profiler.output_sizes = [768.0] * num_layers
            for d in range(self.num_devices):
                self.profiler.bandwidths[d] = 50000.0
                self.profiler.available_memory[d] = 20.0

        self.scheduler = ConfidantScheduler(self.profiler, num_layers, self.num_devices)
        partition_points = self.scheduler.calculate_partition_point(is_average=True)

        # --- Step 2: Create per-device layers ---
        logger.info("\n[2/5] Creating per-device layers...")

        for device_id in range(self.num_devices):
            device = torch.device(
                f"cuda:{device_id}" if torch.cuda.is_available() else "cpu"
            )

            # SEPARATE state per device (own commit tracking for 1F1B)
            state = ConfidantStateManager()
            state.set_device_idx(device_id)
            state.set_worker_num(self.num_devices)
            state.set_partition_point(partition_points)
            state.set_batch_size(self.batch_size)
            workers_map = {d: f"localhost:{50051 + d}" for d in range(self.num_devices)}
            state.set_workers(workers_map)
            self.states[device_id] = state

            # Compute — supports two modes:
            #   defer_step=True:  Gradient accumulation. Step once at epoch end.
            #   defer_step=False: Per-micro-batch updates (mirrors Java Confidant).
            #     Uses forward recomputation during backward to create fresh
            #     autograd graphs, avoiding stale-version errors from in-place
            #     weight updates by earlier micro-batches' optimizer.step().
            cb = ConfidantComputeBackend(device, device_id=device_id,
                                         event_logger=self.event_logger,
                                         defer_step=False)
            sub_model = cb.create_sub_model(full_model, partition_points,
                                            device_id, self.num_devices, device)
            self.computes[device_id] = cb

            # Optimizer
            ob = ConfidantOptimizerBackend()
            opt = ob.create_optimizer(sub_model, self.lr)
            cb.optimizer_ref = opt
            self.optimizers[device_id] = ob

            # Fault tolerance
            ft = ConfidantFaultTolerance(state, self.checkpoint_dir)
            self.fts[device_id] = ft

            # Communication
            self.comm.start_server("worker" if device_id > 0 else "central",
                                   device_id, f"localhost:{50051 + device_id}")

        # --- Step 3: Create roles (each with its own state) ---
        logger.info("\n[3/5] Creating orchestration roles...")

        # Epoch-end callback: step all worker optimizers after all
        # micro-batches complete (central steps itself in start_train)
        def _epoch_end_step_workers(epoch: int):
            for did in range(1, self.num_devices):
                self.computes[did].step_optimizer()
            logger.debug(f"  Worker optimizers stepped for epoch {epoch + 1}")

        self.central = CentralRole(
            state=self.states[0],
            compute=self.computes[0],
            optimizer=self.optimizers[0],
            dataset=self.dataset,
            comm=self.comm,
            ft=self.fts[0],
            event_logger=self.event_logger,
            epoch_end_callback=_epoch_end_step_workers,
        )

        for device_id in range(1, self.num_devices):
            worker = WorkerRole(
                device_id=device_id,
                state=self.states[device_id],  # Worker's OWN state
                compute=self.computes[device_id],
                optimizer=self.optimizers[device_id],
                comm=self.comm,
                ft=self.fts[device_id],
                event_logger=self.event_logger,
            )
            self.workers[device_id] = worker

        # --- Step 4: Register communication handlers ---
        logger.info("\n[4/5] Registering communication handlers...")

        self.comm.register_central_backward_handler(
            self.central.handle_backward_intermediate
        )
        for device_id, worker in self.workers.items():
            self.comm.register_forward_handler(device_id, worker.handle_forward)
            self.comm.register_backward_handler(device_id, worker.handle_backward)
            self.comm.register_label_handler(device_id, worker.receive_labels)
            self.comm.register_epoch_handler(device_id, worker.init_epoch)
            self.comm.register_basic_info_handler(device_id, worker.set_basic_info)

        # --- Step 5: Create dataset ---
        logger.info("\n[5/5] Creating dataset...")
        self.dataset.create_dataset(dataset_name, dataset_path,
                                    self.batch_size, **dataset_kwargs)

        # Enable dashboard
        num_batches = self.dataset.get_data_len()
        if self._enable_dashboard:
            self.event_logger.enable_dashboard(
                num_devices=self.num_devices,
                num_batches=num_batches,
                num_epochs=self.num_epochs,
            )
            logger.info(f"  Live Dashboard: ENABLED ({self.num_devices} devices × {num_batches} batches)")

        logger.info(f"\n{'='*60}")
        logger.info(f"SETUP COMPLETE")
        logger.info(f"  Partition: {partition_points}")
        logger.info(f"  Per-device state (separate commit tracking per device)")
        defer_mode = "accumulate (epoch-end step)" if self.computes[0].defer_step else "per-micro-batch (recompute)"
        logger.info(f"  Gradient mode: defer_step={self.computes[0].defer_step} — {defer_mode}")
        for d in range(self.num_devices):
            cb = self.computes[d]
            logger.info(f"  Device {d}: sub_model on {cb.device}")
        logger.info(f"  Dataset: {num_batches} batches")
        logger.info(f"  Dashboard: {'enabled' if self._enable_dashboard else 'disabled'}")
        logger.info(f"{'='*60}\n")

    def start_train(self, num_epochs: int = None):
        epochs = num_epochs or self.num_epochs

        # Reset error flag from previous runs
        self.comm._error = None

        # Clear stale events from previous runs
        self.event_logger.clear()

        # Start live dashboard
        self.event_logger.start_dashboard()

        try:
            self.central.start_train(epochs)
        finally:
            # Ensure all async comm operations are done
            self.comm.wait_all()
            # Always stop dashboard cleanly
            self.event_logger.stop_dashboard()

        logger.info("\n" + "="*60)
        logger.info("TRAINING COMPLETE")
        logger.info("="*60)

    def export_observability(self, trace_path: str = "1f1b_trace.json",
                              events_path: str = "1f1b_events.json"):
        """Export all observability data after training — Rich formatted."""
        el = self.event_logger

        # 1. Chrome trace
        el.to_chrome_trace(trace_path)

        # 2. Raw events JSON
        el.to_json(events_path)

        # 3. Rich summary table
        CONSOLE.print()
        CONSOLE.rule("[bold bright_white]OBSERVABILITY REPORT[/]", style="bright_blue")
        CONSOLE.print()

        el.print_rich_summary()
        CONSOLE.print()

        # 4. Rich 1F1B timeline
        el.print_rich_1f1b_timeline()
        CONSOLE.print()

        # 5. Rich per-iteration breakdown (first 3 iters)
        el.print_rich_iteration_breakdown(max_iters=3)
        CONSOLE.print()

        # 6. Loss summary
        if el._dashboard and el._dashboard._losses:
            loss_table = Table(
                title="[bold]Loss History (last 10)[/]",
                box=box.ROUNDED,
            )
            loss_table.add_column("Iter", style="yellow", justify="center")
            loss_table.add_column("Loss", justify="right")
            for it, loss in el._dashboard._losses[-10:]:
                color = "green" if loss < 1.0 else "yellow" if loss < 2.0 else "red"
                loss_table.add_row(str(it), f"[{color}]{loss:.4f}[/]")
            CONSOLE.print(loss_table)
            CONSOLE.print()

        CONSOLE.rule("[bold bright_white]END REPORT[/]", style="bright_blue")

        return el.get_events()


logger.info("L8: Orchestrator + ConfidantOrchestrator loaded (separate state + grad accum)")


In [ ]:
# =============================================================================
# TEST: Run BERT Training on 2 GPUs with Live Dashboard + Observability
# =============================================================================

# Define a simple BERT model
class SimpleBERTModel(nn.Module):
    def __init__(self, hidden_size=768, num_layers=12, intermediate_size=3072, num_classes=2):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_size, intermediate_size),
                nn.GELU(),
                nn.Linear(intermediate_size, hidden_size),
                nn.LayerNorm(hidden_size),
            )
            for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x) + x
        return self.classifier(x.mean(dim=1))


# Create model
model = SimpleBERTModel(hidden_size=768, num_layers=12)
logger.info(f"Model: {sum(p.numel() for p in model.parameters()):,} parameters")

# Create orchestrator — dashboard is enabled by default
# Set enable_dashboard=False to disable the live Rich terminal display
orchestrator = ConfidantOrchestrator(
    num_devices=4,
    num_epochs=2,
    batch_size=16,
    lr=1e-4,
    checkpoint_dir="./checkpoints_bert",
    enable_dashboard=True,   # Phase 2: Rich live dashboard
)

# Setup (profiles, partitions, wires everything, creates dashboard)
orchestrator.setup(
    full_model=model,
    dataset_name="dummy",
    profile=True,  # Set False if no GPU
    num_samples=160,
    seq_len=128,
    hidden_dim=768,
    num_classes=2,
)

# Train (dashboard starts, updates live, stops when done)
orchestrator.start_train()

# ── Export Observability (Rich formatted) ────────────────────────────────
# Produces:
#   1f1b_trace.json    → open in chrome://tracing or https://ui.perfetto.dev
#   1f1b_events.json   → raw structured events for analysis
#   Rich terminal       → summary table + 1F1B timeline + per-iter breakdown
events = orchestrator.export_observability(
    trace_path="1f1b_outputs/1f1b_trace.json",
    events_path="1f1b_outputs/1f1b_events.json",
)
logger.info(f"Total events captured: {len(events)}")

In [ ]:
# =============================================================================
# TEST: Run GPT-2 Small Training on 4 GPUs with Live Dashboard + Observability
# =============================================================================

import math

# ── GPT-2 Style Model ────────────────────────────────────────────────────
class GPT2Block(nn.Module):
    """Single GPT-2 transformer block with multi-head self-attention + FFN."""
    def __init__(self, hidden_size=768, num_heads=12, intermediate_size=3072, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(hidden_size)
        self.attn = nn.MultiheadAttention(hidden_size, num_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, intermediate_size),
            nn.GELU(),
            nn.Linear(intermediate_size, hidden_size),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # Self-attention with pre-norm (GPT-2 style)
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + attn_out
        # FFN with pre-norm
        x = x + self.ffn(self.ln2(x))
        return x


class GPT2Model(nn.Module):
    """GPT-2 Small: 12 layers, 768 hidden, 12 heads, ~125M params.
    
    Architecture matches OpenAI GPT-2 (117M) with token + positional embeddings,
    12 transformer blocks, and a language modeling head.
    
    For pipeline parallelism the `layers` attribute wraps each transformer block
    so the partitioner can split them across GPUs.
    """
    def __init__(
        self,
        vocab_size: int = 50257,
        max_seq_len: int = 512,
        hidden_size: int = 768,
        num_layers: int = 12,
        num_heads: int = 12,
        intermediate_size: int = 3072,
        num_classes: int = 2,       # downstream classification head
        dropout: float = 0.1,
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.vocab_size = vocab_size

        # Token + positional embeddings
        self.token_emb = nn.Embedding(vocab_size, hidden_size)
        self.pos_emb = nn.Embedding(max_seq_len, hidden_size)
        self.drop = nn.Dropout(dropout)

        # Transformer blocks — each wrapped in nn.Sequential so the
        # partitioner sees a flat `layers` ModuleList it can slice.
        self.layers = nn.ModuleList([
            nn.Sequential(GPT2Block(hidden_size, num_heads, intermediate_size, dropout))
            for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(hidden_size)

        # Classification head (swap for LM head if doing pretraining)
        self.classifier = nn.Linear(hidden_size, num_classes)

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x):
        """
        x: (batch, seq_len, hidden_size) — pre-embedded input from the pipeline.
        
        NOTE: The pipeline partitioner feeds raw float tensors (not token ids)
        because the dataset generates dummy float data. In a real LM setup you'd
        pass token ids and embed here. For pipeline-parallel compatibility we
        skip the embedding lookup and accept pre-embedded inputs.
        """
        # x is already (batch, seq_len, hidden) from the dummy dataset
        B, T, C = x.shape
        pos_ids = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = x + self.pos_emb(pos_ids)
        x = self.drop(x)

        for layer in self.layers:
            x = layer(x) + x  # residual (matches partitioner's SubModel.forward)

        x = self.ln_f(x)
        return self.classifier(x.mean(dim=1))  # pool → classify


# ── Configure ─────────────────────────────────────────────────────────────
#
# GPT-2 Small: 12 layers, 768 hidden, 12 heads ≈ 125M params
# Dataset: 2048 synthetic samples, seq_len=256 — enough for ~128 batches
# 4 GPUs, 3 epochs, batch_size=16
#
  # adjust to your available GPUs

NUM_DEVICES = 4
NUM_EPOCHS = 3
BATCH_SIZE = 16
LR = 3e-4
HIDDEN_SIZE = 768
NUM_LAYERS = 12
NUM_HEADS = 12
INTERMEDIATE_SIZE = 3072
SEQ_LEN = 256
NUM_SAMPLES = 2048      # 2048 / 16 = 128 batches per epoch
NUM_CLASSES = 2
VOCAB_SIZE = 50257

# Create model
model = GPT2Model(
    vocab_size=VOCAB_SIZE,
    max_seq_len=SEQ_LEN,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    intermediate_size=INTERMEDIATE_SIZE,
    num_classes=NUM_CLASSES,
)
total_params = sum(p.numel() for p in model.parameters())
logger.info(f"GPT-2 Small: {total_params:,} parameters ({total_params/1e6:.1f}M)")

# Create orchestrator
orchestrator = ConfidantOrchestrator(
    num_devices=NUM_DEVICES,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    checkpoint_dir="./checkpoints_gpt2",
    enable_dashboard=True,
)

# Setup (profiles layers on each GPU, partitions across 4 GPUs, wires everything)
orchestrator.setup(
    full_model=model,
    dataset_name="dummy",
    profile=True,
    num_samples=NUM_SAMPLES,
    seq_len=SEQ_LEN,
    hidden_dim=HIDDEN_SIZE,
    num_classes=NUM_CLASSES,
)

# Train
orchestrator.start_train()

# ── Export Observability ──────────────────────────────────────────────────
events = orchestrator.export_observability(
    trace_path="1f1b_outputs/gpt2_trace.json",
    events_path="1f1b_outputs/gpt2_events.json",
)
logger.info(f"Total events captured: {len(events)}")
logger.info(f"Model: GPT-2 Small ({total_params/1e6:.1f}M params)")
logger.info(f"Config: {NUM_DEVICES} GPUs, {NUM_EPOCHS} epochs, "
            f"{NUM_SAMPLES} samples, seq_len={SEQ_LEN}, bs={BATCH_SIZE}")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} → {torch.cuda.get_device_name(i)}")

In [ ]:
import torch
print(f"Device count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    try:
        torch.cuda.reset_peak_memory_stats(i)
        print(f"  cuda:{i} ✓ {torch.cuda.get_device_name(i)}")
    except RuntimeError as e:
        print(f"  cuda:{i} ✗ {e}")

In [ ]:
# =============================================================================
# CLEANUP: Free all GPU memory from BERT run before GPT-2
# =============================================================================
import gc

# Delete BERT orchestrator and all its references
try:
    del orchestrator
    del model
    del events
except NameError:
    pass


# Force Python garbage collection
gc.collect()

# Free all cached GPU memory
for i in range(torch.cuda.device_count()):
    torch.cuda.set_device(i)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(i)
    allocated = torch.cuda.memory_allocated(i) / (1024**3)
    reserved = torch.cuda.memory_reserved(i) / (1024**3)
    print(f"  cuda:{i}: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB")

logger.info("GPU memory cleared — ready for GPT-2 run")

In [ ]:
import subprocess
result = subprocess.run(
    ["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader,nounits"],
    capture_output=True, text=True
)
current_pid = os.getpid()
for line in result.stdout.strip().split('\n'):
    line = line.strip()
    if line and line.isdigit():
        pid = int(line)
        if pid != current_pid:
            # Check if it's a Python process (avoid killing unrelated jobs)
            try:
                cmdline = open(f"/proc/{pid}/cmdline", "r").read()
                if "python" in cmdline.lower() and "dheeraj" in open(f"/proc/{pid}/environ", "r").read():
                    logger.warning(f"Found stale GPU process {pid}, consider: kill -9 {pid}")
            except (FileNotFoundError, PermissionError):
                pass

In [ ]:
import gc, torch

# Kill all references
for name in list(globals()):
    if isinstance(globals()[name], (torch.Tensor, torch.nn.Module)):
        del globals()[name]
gc.collect()

for i in range(torch.cuda.device_count()):
    torch.cuda.set_device(i)
    torch.cuda.synchronize(i)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(i)

In [ ]:
for i in range(4):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}: {free/1e9:.1f}GB free / {total/1e9:.1f}GB total")

In [ ]:
# # =============================================================================
# # GPT-2 Pipeline-Parallel: SST-2 Sentiment Classification (Real Dataset)
# # =============================================================================
# #
# # End-to-end flow:
# #   1. Download SST-2 (Stanford Sentiment Treebank) via HuggingFace datasets
# #   2. Tokenize with GPT-2 tokenizer + embed with pretrained GPT-2 embeddings
# #      → produces (batch, seq_len, 768) float tensors compatible with pipeline
# #   3. Train with Confidant 4-GPU pipeline parallelism
# #   4. Reassemble sub-model weights → single GPU inference
# #   5. Evaluate on validation set + show per-sample predictions
# #
# # Why pre-embed? The pipeline partitioner only slices model.layers (transformer
# # blocks). Embeddings live outside the pipeline. So we embed text offline and
# # feed float tensors — the same approach used for the synthetic dataset, but
# # now with real language data and pretrained embeddings that carry semantic info.
# # =============================================================================

# import gc, math, time

# # ── 0. Install deps (run once) ──────────────────────────────────────────
# # If datasets/transformers aren't installed, uncomment and run:
# # !pip install datasets transformers

# from datasets import load_dataset
# from transformers import GPT2Tokenizer, GPT2Model as HF_GPT2Model

# # ── 1. Load SST-2 ───────────────────────────────────────────────────────

# logger.info("Loading SST-2 dataset from HuggingFace...")
# sst2 = load_dataset("stanfordnlp/sst2")

# # SST-2 has train (67,349) and validation (872) splits
# # Test split labels are hidden (-1), so we use validation as test
# train_raw = sst2["train"]
# val_raw = sst2["validation"]

# logger.info(f"SST-2 loaded: {len(train_raw)} train, {len(val_raw)} validation")
# logger.info(f"Example: '{train_raw[0]['sentence']}' → label={train_raw[0]['label']}")

# # ── 2. Tokenize + Embed ─────────────────────────────────────────────────
# # Use pretrained GPT-2 tokenizer + embedding layer to convert text → floats.
# # We ONLY use the embedding (token_emb + pos_emb) — not the full GPT-2 model.
# # This gives us (batch, seq_len, 768) tensors with real semantic content.

# SEQ_LEN = 128
# HIDDEN_SIZE = 768

# logger.info("Loading GPT-2 tokenizer + embedding layer...")
# tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

# # Load only the embedding weights from pretrained GPT-2
# hf_gpt2 = HF_GPT2Model.from_pretrained("gpt2")
# pretrained_token_emb = hf_gpt2.wte  # token embedding: (50257, 768)
# pretrained_pos_emb = hf_gpt2.wpe    # position embedding: (1024, 768)

# # Move to GPU for fast embedding, then we'll bring results to CPU
# embed_device = torch.device("cuda:0")
# pretrained_token_emb = pretrained_token_emb.to(embed_device)
# pretrained_pos_emb = pretrained_pos_emb.to(embed_device)

# def embed_texts(texts, labels_list, max_len=SEQ_LEN, batch_size=256):
#     """Tokenize and embed a list of texts using pretrained GPT-2 embeddings.
#     Returns: (embeddings_tensor, labels_tensor) on CPU
#     """
#     all_embeds = []
#     all_labels = []
    
#     for start in range(0, len(texts), batch_size):
#         batch_texts = texts[start:start + batch_size]
#         batch_labels = labels_list[start:start + batch_size]
        
#         # Tokenize
#         encoded = tokenizer(
#             batch_texts,
#             padding="max_length",
#             truncation=True,
#             max_length=max_len,
#             return_tensors="pt",
#         )
#         input_ids = encoded["input_ids"].to(embed_device)  # (B, seq_len)
        
#         # Embed: token_emb + pos_emb (no grad needed, just extracting features)
#         with torch.no_grad():
#             pos_ids = torch.arange(max_len, device=embed_device).unsqueeze(0)
#             embeds = pretrained_token_emb(input_ids) + pretrained_pos_emb(pos_ids)
        
#         all_embeds.append(embeds.cpu())
#         all_labels.append(torch.tensor(batch_labels, dtype=torch.long))
        
#         if start % 5000 == 0 and start > 0:
#             logger.info(f"  Embedded {start}/{len(texts)} samples...")
    
#     return torch.cat(all_embeds, dim=0), torch.cat(all_labels, dim=0)


# logger.info("Embedding training set...")
# t0 = time.time()

# # Subsample training set for faster demo (use full set for real training)
# # SST-2 has 67k samples — using 4096 gives ~256 batches which is plenty
# NUM_TRAIN = 4096
# NUM_TEST = len(val_raw)  # use full validation set (872 samples)

# # Shuffle and subsample train set
# train_indices = torch.randperm(len(train_raw))[:NUM_TRAIN].tolist()
# train_texts = [train_raw[i]["sentence"] for i in train_indices]
# train_labels = [train_raw[i]["label"] for i in train_indices]

# train_embeds, train_label_tensor = embed_texts(train_texts, train_labels)
# logger.info(f"  Train: {train_embeds.shape} in {time.time()-t0:.1f}s")

# logger.info("Embedding validation set...")
# t0 = time.time()
# val_texts = [val_raw[i]["sentence"] for i in range(NUM_TEST)]
# val_labels = [val_raw[i]["label"] for i in range(NUM_TEST)]
# val_embeds, val_label_tensor = embed_texts(val_texts, val_labels)
# logger.info(f"  Val: {val_embeds.shape} in {time.time()-t0:.1f}s")

# # Free the pretrained embedding layers
# del hf_gpt2, pretrained_token_emb, pretrained_pos_emb
# torch.cuda.empty_cache()
# gc.collect()

# logger.info(f"Train label distribution: {torch.bincount(train_label_tensor).tolist()} (neg/pos)")
# logger.info(f"Val label distribution:   {torch.bincount(val_label_tensor).tolist()} (neg/pos)")


In [ ]:

# =============================================================================
# GPT-2 Pipeline-Parallel: SST-2 Sentiment Classification (Real Dataset)
# =============================================================================
#
# End-to-end flow:
#   1. Download SST-2 (Stanford Sentiment Treebank) via HuggingFace datasets
#   2. Tokenize with GPT-2 tokenizer + embed with pretrained GPT-2 embeddings
#      → produces (batch, seq_len, 768) float tensors compatible with pipeline
#   3. Train with Confidant 4-GPU pipeline parallelism
#   4. Reassemble sub-model weights → single GPU inference
#   5. Evaluate on validation set + show per-sample predictions
#
# Why pre-embed? The pipeline partitioner only slices model.layers (transformer
# blocks). Embeddings live outside the pipeline. So we embed text offline and
# feed float tensors — the same approach used for the synthetic dataset, but
# now with real language data and pretrained embeddings that carry semantic info.
# =============================================================================

import gc, math, time

# ── 0. Install deps (run once) ──────────────────────────────────────────
# If datasets/transformers aren't installed, uncomment and run:
# !pip install datasets transformers

from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2Model as HF_GPT2Model

# ── 1. Load SST-2 ───────────────────────────────────────────────────────

logger.info("Loading SST-2 dataset from HuggingFace...")
sst2 = load_dataset("stanfordnlp/sst2")

# SST-2 has train (67,349) and validation (872) splits
# Test split labels are hidden (-1), so we use validation as test
train_raw = sst2["train"]
val_raw = sst2["validation"]

logger.info(f"SST-2 loaded: {len(train_raw)} train, {len(val_raw)} validation")
logger.info(f"Example: '{train_raw[0]['sentence']}' → label={train_raw[0]['label']}")

# ── 2. Tokenize + Embed ─────────────────────────────────────────────────
# Use pretrained GPT-2 tokenizer + embedding layer to convert text → floats.
# We ONLY use the embedding (token_emb + pos_emb) — not the full GPT-2 model.
# This gives us (batch, seq_len, 768) tensors with real semantic content.

SEQ_LEN = 128
HIDDEN_SIZE = 768

logger.info("Loading GPT-2 tokenizer + embedding layer...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

# Load only the embedding weights from pretrained GPT-2
hf_gpt2 = HF_GPT2Model.from_pretrained("gpt2")
pretrained_token_emb = hf_gpt2.wte  # token embedding: (50257, 768)
pretrained_pos_emb = hf_gpt2.wpe    # position embedding: (1024, 768)

# Move to GPU for fast embedding, then we'll bring results to CPU
embed_device = torch.device("cuda:0")
pretrained_token_emb = pretrained_token_emb.to(embed_device)
pretrained_pos_emb = pretrained_pos_emb.to(embed_device)

def embed_texts(texts, labels_list, max_len=SEQ_LEN, batch_size=256):
    """Tokenize and embed a list of texts using pretrained GPT-2 embeddings.
    Returns: (embeddings_tensor, labels_tensor) on CPU
    """
    all_embeds = []
    all_labels = []
    
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        batch_labels = labels_list[start:start + batch_size]
        
        # Tokenize
        encoded = tokenizer(
            batch_texts,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        input_ids = encoded["input_ids"].to(embed_device)  # (B, seq_len)
        
        # Embed: token_emb + pos_emb (no grad needed, just extracting features)
        with torch.no_grad():
            pos_ids = torch.arange(max_len, device=embed_device).unsqueeze(0)
            embeds = pretrained_token_emb(input_ids) + pretrained_pos_emb(pos_ids)
        
        all_embeds.append(embeds.cpu())
        all_labels.append(torch.tensor(batch_labels, dtype=torch.long))
        
        if start % 5000 == 0 and start > 0:
            logger.info(f"  Embedded {start}/{len(texts)} samples...")
    
    return torch.cat(all_embeds, dim=0), torch.cat(all_labels, dim=0)


logger.info("Embedding training set...")
t0 = time.time()

# Subsample training set for faster demo (use full set for real training)
# SST-2 has 67k samples — using 4096 gives ~256 batches which is plenty
NUM_TRAIN = 4096
NUM_TEST = len(val_raw)  # use full validation set (872 samples)

# Shuffle and subsample train set
train_indices = torch.randperm(len(train_raw))[:NUM_TRAIN].tolist()
train_texts = [train_raw[i]["sentence"] for i in train_indices]
train_labels = [train_raw[i]["label"] for i in train_indices]

train_embeds, train_label_tensor = embed_texts(train_texts, train_labels)
logger.info(f"  Train: {train_embeds.shape} in {time.time()-t0:.1f}s")

logger.info("Embedding validation set...")
t0 = time.time()
val_texts = [val_raw[i]["sentence"] for i in range(NUM_TEST)]
val_labels = [val_raw[i]["label"] for i in range(NUM_TEST)]
val_embeds, val_label_tensor = embed_texts(val_texts, val_labels)
logger.info(f"  Val: {val_embeds.shape} in {time.time()-t0:.1f}s")

# Free the pretrained embedding layers
del hf_gpt2, pretrained_token_emb, pretrained_pos_emb
torch.cuda.empty_cache()
gc.collect()

logger.info(f"Train label distribution: {torch.bincount(train_label_tensor).tolist()} (neg/pos)")
logger.info(f"Val label distribution:   {torch.bincount(val_label_tensor).tolist()} (neg/pos)")



# ── 3. Define GPT-2 Model ───────────────────────────────────────────────
# Same architecture as cell 11, but adapted:
#   - No token embedding (we pre-embed)
#   - dropout=0.1 for regularization on real data
#   - classifier head for binary sentiment

class GPT2Block(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12, intermediate_size=3072, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(hidden_size)
        self.attn = nn.MultiheadAttention(hidden_size, num_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, intermediate_size),
            nn.GELU(),
            nn.Linear(intermediate_size, hidden_size),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x


class GPT2Classifier(nn.Module):
    """GPT-2 style transformer for classification on pre-embedded inputs.
    
    Input: (batch, seq_len, hidden_size) — already embedded by pretrained GPT-2.
    Output: (batch, num_classes) — sentiment logits.
    
    The `layers` attribute is an nn.ModuleList so the Confidant partitioner
    can split transformer blocks across GPUs.
    """
    def __init__(self, max_seq_len=128, hidden_size=768, num_layers=12,
                 num_heads=12, intermediate_size=3072, num_classes=2, dropout=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        
        # Learnable positional refinement (input already has pretrained pos emb,
        # but a small learned adjustment helps fine-tuning)
        self.pos_emb = nn.Embedding(max_seq_len, hidden_size)
        self.drop = nn.Dropout(dropout)
        
        # Transformer blocks
        self.layers = nn.ModuleList([
            nn.Sequential(GPT2Block(hidden_size, num_heads, intermediate_size, dropout))
            for _ in range(num_layers)
        ])
        
        self.ln_f = nn.LayerNorm(hidden_size)
        self.classifier = nn.Linear(hidden_size, num_classes)
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x):
        B, T, C = x.shape
        pos_ids = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = x + self.pos_emb(pos_ids)
        x = self.drop(x)
        for layer in self.layers:
            x = layer(x)  # <--- FIXED: Removed "+ x"
        x = self.ln_f(x)
        return self.classifier(x.mean(dim=1))


# ── 4. Setup Orchestrator + Inject Real Data ─────────────────────────────
# ── 4. Setup Orchestrator + Inject Real Data ─────────────────────────────

BATCH_SIZE = 16
NUM_EPOCHS = 20
LR = 3e-4
NUM_DEVICES = 4

model = GPT2Classifier(
    max_seq_len=SEQ_LEN,
    hidden_size=HIDDEN_SIZE,
    num_layers=12,
    num_heads=12,
    intermediate_size=3072,
    num_classes=2,
    dropout=0.1,
)
total_params = sum(p.numel() for p in model.parameters())
logger.info(f"GPT-2 Classifier: {total_params:,} parameters ({total_params/1e6:.1f}M)")

orchestrator = ConfidantOrchestrator(
    num_devices=NUM_DEVICES,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    checkpoint_dir="./checkpoints_sst2",
    enable_dashboard=True,
)

# Setup with dummy data (for profiling + partition), then swap in real data
orchestrator.setup(
    full_model=model,
    dataset_name="dummy",
    profile=True,
    num_samples=NUM_TRAIN,
    seq_len=SEQ_LEN,
    hidden_dim=HIDDEN_SIZE,
    num_classes=2,
)

# Swap in real SST-2 embedded data
train_dataset = TensorDataset(train_embeds, train_label_tensor)
orchestrator.dataset.dataset = train_dataset
orchestrator.dataset.dataloader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True
)
orchestrator.dataset._data_len = len(orchestrator.dataset.dataloader)
logger.info(f"Injected SST-2 data: {len(train_dataset)} samples, "
            f"{orchestrator.dataset._data_len} batches/epoch")
accumulation_steps = len(orchestrator.dataset.dataloader)
logger.info(f"Configuring backends with accumulation_steps={accumulation_steps}")
# ── 5. Train ─────────────────────────────────────────────────────────────
for device_id, backend in orchestrator.computes.items():
    if hasattr(backend, 'accumulation_steps'):
        backend.accumulation_steps = accumulation_steps
logger.info("\n" + "=" * 60)
logger.info("TRAINING: GPT-2 on SST-2 Sentiment (4-GPU Pipeline)")
logger.info("=" * 60)

orchestrator.start_train()

events = orchestrator.export_observability(
    trace_path="1f1b_outputs/sst2_trace.json",
    events_path="1f1b_outputs/sst2_events.json",
)

# ── 6. SAVE: Gather and Save to Disk (NEW) ───────────────────────────────

logger.info("\n" + "=" * 60)
logger.info("SAVING: Gathering distributed weights to single file")
logger.info("=" * 60)

# Create a container model on CPU to hold the gathered weights
cpu_model = GPT2Classifier(
    max_seq_len=SEQ_LEN,
    hidden_size=HIDDEN_SIZE,
    num_layers=12,
    num_heads=12,
    intermediate_size=3072,
    num_classes=2,
    dropout=0.1, 
)

partition_points = orchestrator.states[0].get_partition_point()
num_layers = 12

for device_id in range(NUM_DEVICES):
    cb = orchestrator.computes[device_id]
    sub_model = cb.sub_model
    
    # Identify which global layers this GPU holds
    if device_id == 0:
        start = 0
        end = partition_points[0] if partition_points else num_layers - 1
    elif device_id < len(partition_points):
        start = partition_points[device_id - 1] + 1
        end = partition_points[device_id]
    else:
        start = partition_points[-1] + 1 if partition_points else 0
        end = num_layers - 1
    
    # Copy Layer Weights: sub_model (local) -> cpu_model (global)
    sub_layers = list(sub_model.layers)
    for i, global_idx in enumerate(range(start, end + 1)):
        src_state = sub_layers[i].state_dict()
        cpu_model.layers[global_idx].load_state_dict(src_state)
    
    # Copy Classifier Weights (Last stage only)
    if sub_model.is_last_stage and sub_model.classifier is not None:
        cpu_model.classifier.load_state_dict(sub_model.classifier.state_dict())
        logger.info(f"  Stage {device_id}: Restored classifier")

    logger.info(f"  Stage {device_id}: Restored layers {start}-{end}")

# Save to disk
save_path = "gpt2_sst2_finetuned.pt"
torch.save(cpu_model.state_dict(), save_path)
logger.info(f"Full model saved to: {save_path}")

# Cleanup to prove we load from disk
del cpu_model, orchestrator
torch.cuda.empty_cache()
gc.collect()




In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from rich.console import Console
from rich.table import Table
from rich import box
from rich.progress import track
import logging

# Setup basic logging if not already done
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(name)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)
CONSOLE = Console()

# ── RE-DEFINE CONFIG (Required since kernel was cleared) ─────────────────
SEQ_LEN = 128
HIDDEN_SIZE = 768
save_path = "gpt2_sst2_finetuned.pt"
inference_device = torch.device("cuda:0")

# ── RE-DEFINE MODEL CLASS ────────────────────────────────────────────────
class GPT2Block(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12, intermediate_size=3072, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(hidden_size)
        self.attn = nn.MultiheadAttention(hidden_size, num_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(hidden_size)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, intermediate_size),
            nn.GELU(),
            nn.Linear(intermediate_size, hidden_size),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x

class GPT2Classifier(nn.Module):
    def __init__(self, max_seq_len=128, hidden_size=768, num_layers=12,
                 num_heads=12, intermediate_size=3072, num_classes=2, dropout=0.1):
        super().__init__()
        self.pos_emb = nn.Embedding(max_seq_len, hidden_size)
        self.drop = nn.Dropout(dropout)
        self.layers = nn.ModuleList([
            nn.Sequential(GPT2Block(hidden_size, num_heads, intermediate_size, dropout))
            for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(hidden_size)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        B, T, C = x.shape
        pos_ids = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = x + self.pos_emb(pos_ids)
        x = self.drop(x)
        for layer in self.layers:
            x = layer(x) 
        x = self.ln_f(x)
        return self.classifier(x.mean(dim=1))

# ── LOAD MODEL & INFERENCE ───────────────────────────────────────────────

logger.info("\n" + "=" * 60)
logger.info("INFERENCE: Loading from disk")
logger.info("=" * 60)

# Instantiate fresh model
infer_model = GPT2Classifier(
    max_seq_len=SEQ_LEN,
    hidden_size=HIDDEN_SIZE,
    num_layers=12,
    num_heads=12,
    intermediate_size=3072,
    num_classes=2,
    dropout=0.0, # Disable dropout for inference
)

# Load weights
try:
    state_dict = torch.load(save_path, map_location='cpu')
    infer_model.load_state_dict(state_dict)
    infer_model = infer_model.to(inference_device)
    infer_model.eval()
    logger.info(f"Model loaded successfully from {save_path}")
except FileNotFoundError:
    logger.error(f"File not found: {save_path}. Make sure training completed and saved the file.")
    raise

# Run Validation
logger.info("\nRunning inference on SST-2 validation set...")

# Ensure val_embeds and val_label_tensor exist (from previous cell)
if 'val_embeds' not in locals() or 'val_label_tensor' not in locals():
    logger.error("Validation data not found! Please run the embedding cell (Step 2) first.")
    raise ValueError("Missing validation data")

val_loader = DataLoader(
    TensorDataset(val_embeds, val_label_tensor),
    batch_size=32,
    shuffle=False,
)

all_preds = []
all_labels_list = []
all_probs = []
total_loss = 0.0
num_batches = 0

with torch.no_grad():
    for inputs, labels in track(val_loader, description="Inferring..."):
        inputs = inputs.to(inference_device)
        labels = labels.to(inference_device)
        
        logits = infer_model(inputs)
        loss = F.cross_entropy(logits, labels)
        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)
        
        all_preds.append(preds.cpu())
        all_labels_list.append(labels.cpu())
        all_probs.append(probs.cpu())
        total_loss += loss.item()
        num_batches += 1

all_preds = torch.cat(all_preds)
all_labels_t = torch.cat(all_labels_list)
all_probs = torch.cat(all_probs)
accuracy = (all_preds == all_labels_t).float().mean().item()
avg_loss = total_loss / num_batches

# ── RESULTS DISPLAY ──────────────────────────────────────────────────────

logger.info("\n" + "=" * 60)
logger.info("SST-2 RESULTS")
logger.info("=" * 60)
logger.info(f"  Validation samples: {len(all_labels_t)}")
logger.info(f"  Validation loss:    {avg_loss:.4f}")
logger.info(f"  Validation acc:     {accuracy:.1%}")

for c, name in enumerate(["Negative", "Positive"]):
    mask = all_labels_t == c
    if mask.sum() > 0:
        class_acc = (all_preds[mask] == c).float().mean().item()
        logger.info(f"  {name} accuracy: {class_acc:.1%} ({mask.sum().item()} samples)")

# Show real example predictions
logger.info(f"\nSample predictions on real movie reviews:")
logger.info(f"  {'':>3} | {'True':>8} | {'Pred':>8} | {'Conf':>5} | Sentence")
logger.info(f"  {'-'*3}-+-{'-'*8}-+-{'-'*8}-+-{'-'*5}-+-{'-'*50}")

correct_mask = all_preds == all_labels_t
wrong_mask = ~correct_mask
show_indices = []
if correct_mask.sum() > 0:
    show_indices.extend(correct_mask.nonzero(as_tuple=True)[0][:5].tolist())
if wrong_mask.sum() > 0:
    show_indices.extend(wrong_mask.nonzero(as_tuple=True)[0][:5].tolist())

label_names = {0: "Negative", 1: "Positive"}

for idx in show_indices[:10]:
    true_label = all_labels_t[idx].item()
    pred_label = all_preds[idx].item()
    conf = all_probs[idx, pred_label].item()
    mark = "✓" if true_label == pred_label else "✗"
    # Ensure val_texts exists
    if 'val_texts' in locals():
        sentence = val_texts[idx][:60] + ("..." if len(val_texts[idx]) > 60 else "")
    else:
        sentence = "[Text data missing]"
        
    logger.info(f"  {mark:>3} | {label_names[true_label]:>8} | {label_names[pred_label]:>8} | "
                f"{conf:>4.0%} | {sentence}")

# ── RICH TABLE SUMMARY ───────────────────────────────────────────────────

results_table = Table(
    title="[bold]SST-2 Sentiment Classification — Inference Results[/]",
    box=box.DOUBLE_EDGE,
    show_lines=True,
)
results_table.add_column("Metric", style="cyan bold", width=25)
results_table.add_column("Value", style="white", width=35)

results_table.add_row("Dataset", "SST-2 (Validation Set)")
results_table.add_row("Model", "GPT-2 Classifier (Finetuned)")
results_table.add_row("", "")
results_table.add_row("[bold green]Val Loss[/]", f"[bold]{avg_loss:.4f}[/]")
results_table.add_row("[bold green]Val Accuracy[/]", f"[bold]{accuracy:.1%}[/]")
results_table.add_row("Negative Acc", f"{(all_preds[all_labels_t==0]==0).float().mean().item():.1%}")
results_table.add_row("Positive Acc", f"{(all_preds[all_labels_t==1]==1).float().mean().item():.1%}")

CONSOLE.print()
CONSOLE.print(results_table)

logger.info("\nDone! Inference complete.")